In [1]:
#!/usr/bin/env python3
"""
Multilingual PCFG toy model — v8
=================================
Adds L_E (invariant DET + pre-nominal ADJ) to the v7 grammar (L_A/B/C/D).

Why: v7 found that ε never binds for L_C anywhere in this grammar —
at n_after_det L_C's optimal output is "uniform over 24 nouns" (doesn't
need gender), and at adj_agree (post-nominal) L_C has already seen the
noun. No position forces L_C to commit to gender without having it.

L_E closes that gap: invariant DET (like L_C, zero gender signal) +
pre-nominal ADJ (like L_D's word order). At the adj_agree_pre position,
L_E must produce a gendered adjective having seen only an uninformative
DET, with the noun not yet generated. Chance = 0.5 (2 genders). This is
the one position in the whole grammar where a language is actually
forced to guess gender blind.

Compare L_E's adj_agree_pre to L_D's adj_agree_pre (same word order,
but L_D's DET DOES mark gender — near-ceiling expected).

Predicted signature per the theory (NOT "L_E's own accuracy moves"):
  - L_E's adj_agree_pre accuracy should stay pinned near 0.5 at BOTH
    10K and 1M (ε doesn't shrink with capacity — A2).
  - What SHOULD move with capacity is the D–E sharing/dissociation
    score on this sub-computation: below N*, pooling with L_D may
    still be worth it despite the quality loss; above N*, the model
    should build L_E its own (still-at-chance) machinery and stop
    borrowing L_D's approach. Look for D–E Pearson dissociation
    emerging with scale while L_E's raw accuracy stays flat.
  - If L_E's accuracy stays flat AND the D–E sharing score also stays
    flat: N* for this pair sits outside [10K, 1M] — not evidence
    against the theory. Would need a wider capacity sweep first.

Baked-in lessons from the L_D detour (v7) — applied from the start:
  1. Annotate branch membership at sampler-writing time, not after.
  2. After any sampler edit, re-run the cell that DEFINES sample_np /
     sample_sentence / sample_packed_sequence, not just the eval cell.
  3. Measure accuracy per sub-computation directly; treat Pearson
     activation-profile as secondary/complementary, not primary.
  4. Report Wilson CIs on every accuracy number; check sample sizes
     per branch before reading anything into a comparison.
  5. Rerun with at least one more seed before trusting any single
     striking number — built into run_sweep() from the outset here.

Leakage confound check (before trusting ANY L_E number): verify that
nothing upstream of the adj_agree_pre position (subject, verb, earlier
NPs, DET, NUM) correlates with the upcoming noun's gender for L_E.
See Cell 3.5.

Cell boundaries marked with # ══════ CELL N ══════
Paste each cell into a notebook.
"""

# ══════════════════════════════════════════════════════════════════════


'\nMultilingual PCFG toy model — v8\n=================================\nAdds L_E (invariant DET + pre-nominal ADJ) to the v7 grammar (L_A/B/C/D).\n\nWhy: v7 found that ε never binds for L_C anywhere in this grammar —\nat n_after_det L_C\'s optimal output is "uniform over 24 nouns" (doesn\'t\nneed gender), and at adj_agree (post-nominal) L_C has already seen the\nnoun. No position forces L_C to commit to gender without having it.\n\nL_E closes that gap: invariant DET (like L_C, zero gender signal) +\npre-nominal ADJ (like L_D\'s word order). At the adj_agree_pre position,\nL_E must produce a gendered adjective having seen only an uninformative\nDET, with the noun not yet generated. Chance = 0.5 (2 genders). This is\nthe one position in the whole grammar where a language is actually\nforced to guess gender blind.\n\nCompare L_E\'s adj_agree_pre to L_D\'s adj_agree_pre (same word order,\nbut L_D\'s DET DOES mark gender — near-ceiling expected).\n\nPredicted signature per the theory (NOT "

In [1]:
# CELL 1 — Setup
# ══════════════════════════════════════════════════════════════════════

import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import math, random, time, json, itertools
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Set
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 16  # tune: with prefetch_factor=4 (see make_loader) this
                  # queues ~64 batches ahead across workers. On a 40-core
                  # machine this should fully hide the ~44ms/batch PCFG
                  # sampling cost behind training compute. Going much
                  # higher than ~16-24 tends to add process/IPC overhead
                  # without further gain for an item this cheap to
                  # generate — worth a quick sweep (see benchmark cell
                  # after Cell 8) rather than assuming more is better.
print(f"Device: {device}  |  NUM_WORKERS: {NUM_WORKERS}  (cpu_count={os.cpu_count()})")


# ══════════════════════════════════════════════════════════════════════


Device: cuda  |  NUM_WORKERS: 16  (cpu_count=40)


In [2]:
# CELL 2 — Language definitions + vocabulary
# ══════════════════════════════════════════════════════════════════════
#
# Design:
#   Gender (M/F) is an inherent property of each noun.
#   ADJ must agree with head N in gender — ALL languages.
#   V agrees with subject NP in number — ALL languages.
#
# ε asymmetry (gender), by language:
#   L_A: DET marks gender directly → cheap to read.               DET_g NUM (ADJ post-N)
#   L_B: DET fuses gender × number → must factor to read gender.  DET_gn (ADJ post-N)
#   L_C: DET is invariant → gender only from noun identity.       DET NUM (ADJ post-N)
#   L_D: DET marks gender directly, but ADJ is PRE-nominal.       DET_g NUM (ADJ pre-N)
#   L_E: DET is invariant, AND ADJ is PRE-nominal.                DET NUM (ADJ pre-N)   <- NEW
#
# L_E is the genuine ε test: at the ADJ position it has NEITHER a
# DET gender cue (unlike L_D) NOR an already-generated noun to copy
# from (unlike L_C's post-nominal case). Chance = 0.5 on the gender
# feature. Compare directly against L_D's adj_agree_pre (near-ceiling
# expected, since L_D's DET does mark gender).

LANGS = ["A", "B", "C", "D", "E"]

N_NOUN_PER_GENDER = 12   # 12 M + 12 F = 24 nouns per language
N_ADJ_LEMMAS = 8         # × 2 gender forms = 16 adj tokens per language
N_VTR_LEMMAS = 6         # × 2 number forms = 12 vtr tokens per language
N_VINTR_LEMMAS = 4       # × 2 number forms = 8 vintr tokens per language
N_PREPS = 4

# ── Build per-language lexicons (generic over LANGS, so L_E is free) ──

def make_nouns(lang):
    return {
        'M': [f'nm_{lang}{i}' for i in range(N_NOUN_PER_GENDER)],
        'F': [f'nf_{lang}{i}' for i in range(N_NOUN_PER_GENDER)],
    }

def make_adjs(lang):
    return {
        'M': [f'adjm_{lang}{i}' for i in range(N_ADJ_LEMMAS)],
        'F': [f'adjf_{lang}{i}' for i in range(N_ADJ_LEMMAS)],
    }

def make_verbs_tr(lang):
    return {
        'SG': [f'vts_{lang}{i}' for i in range(N_VTR_LEMMAS)],
        'PL': [f'vtp_{lang}{i}' for i in range(N_VTR_LEMMAS)],
    }

def make_verbs_intr(lang):
    return {
        'SG': [f'vis_{lang}{i}' for i in range(N_VINTR_LEMMAS)],
        'PL': [f'vip_{lang}{i}' for i in range(N_VINTR_LEMMAS)],
    }

NOUNS = {L: make_nouns(L) for L in LANGS}
ADJS  = {L: make_adjs(L) for L in LANGS}
VTR   = {L: make_verbs_tr(L) for L in LANGS}
VINTR = {L: make_verbs_intr(L) for L in LANGS}
PREPS = {L: [f'p_{L}{i}' for i in range(N_PREPS)] for L in LANGS}
RPROS = {L: f'rp_{L}' for L in LANGS}
CONJS = {L: f'cj_{L}' for L in LANGS}

# Determiners — the ε-critical tokens
DETS_A = {'M': 'dtm_A', 'F': 'dtf_A'}
DETS_B = {('M','SG'): 'dtms_B', ('M','PL'): 'dtmp_B',
          ('F','SG'): 'dtfs_B', ('F','PL'): 'dtfp_B'}
DET_C  = 'dt_C'
DETS_D = {'M': 'dtm_D', 'F': 'dtf_D'}  # same gender-marking cost as L_A
DET_E  = 'dt_E'                        # invariant, like L_C — NEW

# Number particles (L_A, L_C, L_D, L_E — L_B fuses number into DET)
NUMS_A = {'SG': 'ns_A', 'PL': 'np_A'}
NUMS_C = {'SG': 'ns_C', 'PL': 'np_C'}
NUMS_D = {'SG': 'ns_D', 'PL': 'np_D'}
NUMS_E = {'SG': 'ns_E', 'PL': 'np_E'}   # NEW — number is the ε≈0 control;
                                        # L_E should differ from L_C ONLY
                                        # in ADJ word order, not in how
                                        # number is encoded.

# Structural / shared
DOT = '.'
BOS, EOS, PAD = '<BOS>', '<EOS>', '<PAD>'
LANG_TOK = {L: f'<L{L}>' for L in LANGS}

# ── Noun → gender lookup (ground truth) ──
NOUN_GENDER: Dict[str, str] = {}
for L in LANGS:
    for g in ['M', 'F']:
        for n in NOUNS[L][g]:
            NOUN_GENDER[n] = g

# ── ADJ lemma index: adj token → lemma index (for measuring agreement) ──
ADJ_LEMMA: Dict[str, int] = {}
for L in LANGS:
    for g in ['M', 'F']:
        for i, tok in enumerate(ADJS[L][g]):
            ADJ_LEMMA[tok] = i

# ── Build full vocabulary ──
ALL_TOKENS: Set[str] = set()

for L in LANGS:
    for g in ['M', 'F']:
        ALL_TOKENS.update(NOUNS[L][g])
        ALL_TOKENS.update(ADJS[L][g])
    for n in ['SG', 'PL']:
        ALL_TOKENS.update(VTR[L][n])
        ALL_TOKENS.update(VINTR[L][n])
    ALL_TOKENS.update(PREPS[L])
    ALL_TOKENS.add(RPROS[L])
    ALL_TOKENS.add(CONJS[L])

ALL_TOKENS.update(DETS_A.values())
ALL_TOKENS.update(DETS_B.values())
ALL_TOKENS.add(DET_C)
ALL_TOKENS.update(DETS_D.values())
ALL_TOKENS.add(DET_E)                  # NEW
ALL_TOKENS.update(NUMS_A.values())
ALL_TOKENS.update(NUMS_C.values())
ALL_TOKENS.update(NUMS_D.values())
ALL_TOKENS.update(NUMS_E.values())     # NEW

ALL_TOKENS.update([DOT, BOS, EOS, PAD])
ALL_TOKENS.update(LANG_TOK.values())

VOCAB = sorted(ALL_TOKENS)
VOCAB_SIZE = len(VOCAB)
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for w, i in STOI.items()}

MAX_LEN = 128  # packed sequence length

print(f"Vocabulary: {VOCAB_SIZE} tokens")
print(f"Nouns per language: {N_NOUN_PER_GENDER * 2}")
print(f"Languages: {LANGS}")
print(f"Max sequence length: {MAX_LEN}")


# ══════════════════════════════════════════════════════════════════════


Vocabulary: 357 tokens
Nouns per language: 24
Languages: ['A', 'B', 'C', 'D', 'E']
Max sequence length: 128


In [3]:
# CELL 3 — PCFG sampler with agreement + position annotations
# ══════════════════════════════════════════════════════════════════════
#
# Grammar (shared non-terminal structure, language-specific terminals):
#
#   S      → CLAUSE  |  CLAUSE CONJ CLAUSE
#   CLAUSE → NP_subj VP
#   NP     → DET_region N [ADJ]  |  DET_region N [ADJ] PP
#            |  DET_region N REL
#   VP     → V_INTR  |  V_TR NP_obj
#   PP     → PREP NP
#   REL    → RPRO VP
#
# DET_region is language-specific:
#   L_A: DET_gender NUM               (ADJ post-nominal)
#   L_B: DET_gender_number            (ADJ post-nominal)
#   L_C: DET_invariant NUM            (ADJ post-nominal)
#   L_D: DET_gender NUM               (ADJ PRE-nominal)
#   L_E: DET_invariant NUM            (ADJ PRE-nominal)   <- NEW
#
# Agreement enforced at generation time:
#   DET agrees with N in gender (L_A, L_D) or gender×number (L_B)
#   ADJ agrees with N in gender (all languages)
#   V agrees with subject NP in number (all languages)
#
# Position annotations:
#   Each token gets a 'subcomp' label for measurement.

# ── Expansion probabilities ──
P_COMPOUND    = 0.4    # S → CLAUSE CONJ CLAUSE
P_ADJ         = 0.40   # NP includes ADJ
P_PP          = 0.15   # NP includes PP (only if no ADJ or stacked)
P_REL         = 0.10   # NP includes REL clause
P_TRANSITIVE  = 0.65   # VP → V_TR NP

MAX_PP_DEPTH  = 2      # prevent infinite PP recursion
MAX_REL_DEPTH = 1      # prevent nested REL clauses


@dataclass
class NPFeatures:
    """Tracks gender/number of the current NP for agreement."""
    gender: str    # 'M' or 'F'
    number: str    # 'SG' or 'PL'
    noun_token: str
    noun_pos: int = -1   # position in the token list where N appears


def sample_sentence(lang: str, rng: random.Random,
                    pp_depth: int = 0, rel_depth: int = 0):
    """
    Sample one sentence from the grammar.
    Returns (tokens, annotations) where annotations[i] is a dict
    with at least {'subcomp': str} for the token at position i.
    """
    tokens = []
    annots = []

    def emit(tok, subcomp, **extra):
        tokens.append(tok)
        ann = {'subcomp': subcomp}
        ann.update(extra)
        annots.append(ann)

    def sample_np(is_subject: bool):
        """Generate an NP, return its NPFeatures."""
        # Gender and number are sampled independently of everything else
        # in the sentence — this is the load-bearing independence
        # property the leakage audit (Cell 3.5) verifies empirically.
        gender = rng.choice(['M', 'F'])
        number = rng.choice(['SG', 'PL'])
        noun = rng.choice(NOUNS[lang][gender])

        # ── DET region (language-specific) ──
        if lang == 'A':
            det_tok = DETS_A[gender]
            emit(det_tok, 'det', gender=gender, number=number)
            num_tok = NUMS_A[number]
            emit(num_tok, 'num', number=number)
        elif lang == 'B':
            det_tok = DETS_B[(gender, number)]
            emit(det_tok, 'det', gender=gender, number=number)
        elif lang == 'C':
            emit(DET_C, 'det', gender=None, number=None)
            num_tok = NUMS_C[number]
            emit(num_tok, 'num', number=number)
        elif lang == 'D':
            # Same DET cost as L_A, but ADJ (if any) comes BEFORE N.
            det_tok = DETS_D[gender]
            emit(det_tok, 'det', gender=gender, number=number)
            num_tok = NUMS_D[number]
            emit(num_tok, 'num', number=number)
        else:  # E — invariant DET (like C) AND pre-nominal ADJ (like D)
            emit(DET_E, 'det', gender=None, number=None)
            num_tok = NUMS_E[number]
            emit(num_tok, 'num', number=number)

        # ── Branch decision (shared across languages) ──
        # For L_D/L_E this must be decided before the noun is emitted,
        # since ADJ (if present) is generated pre-nominally.
        r = rng.random()
        has_adj = (r < P_ADJ)
        has_pp = (not has_adj and r < P_ADJ + P_PP and pp_depth < MAX_PP_DEPTH)
        has_rel = (not has_adj and not has_pp and r < P_ADJ + P_PP + P_REL
                   and rel_depth < MAX_REL_DEPTH)

        if lang in ('D', 'E') and has_adj:
            # ── Pre-nominal ADJ: THE key sub-computation. ──
            # L_D: gender comes from DET_D (available, cheap to read).
            # L_E: DET_E is invariant — gender has NO cheap source here,
            # and the noun doesn't exist in context yet, so this can't
            # be solved by copying the noun's gender the way
            # post-nominal ADJ can. This is the genuine ε test.
            lemma_idx = rng.randrange(N_ADJ_LEMMAS)
            adj_tok = ADJS[lang][gender][lemma_idx]
            emit(adj_tok, 'adj_agree_pre',
                 target_gender=gender, lemma=lemma_idx,
                 head_noun_pos=None)  # noun doesn't exist yet

        # ── Noun ──
        noun_pos = len(tokens)
        # For L_A/L_B/L_D: gender was already signaled by DET.
        # For L_C/L_E: THIS is where gender first becomes available
        # (from noun identity). subcomp 'n_after_det': model predicts
        # N; constrained by DET gender wherever DET marks it (A, B, D),
        # unconstrained for C/E.
        emit(noun, 'n_after_det', gender=gender, number=number)

        npf = NPFeatures(gender=gender, number=number,
                         noun_token=noun, noun_pos=noun_pos)

        if lang not in ('D', 'E'):
            if has_adj:
                lemma_idx = rng.randrange(N_ADJ_LEMMAS)
                adj_tok = ADJS[lang][gender][lemma_idx]
                emit(adj_tok, 'adj_agree',
                     target_gender=gender, lemma=lemma_idx,
                     head_noun_pos=npf.noun_pos)

                # Sometimes also a PP after ADJ
                if rng.random() < P_PP and pp_depth < MAX_PP_DEPTH:
                    sample_pp()

            elif has_pp:
                sample_pp()
            elif has_rel:
                sample_rel(npf)
        else:
            # L_D/L_E: adj already emitted above (pre-nominal) if has_adj.
            # PP/REL branches still apply post-nominally as usual.
            if has_pp:
                sample_pp()
            elif has_rel:
                sample_rel(npf)

        return npf

    def sample_vp(subj: NPFeatures):
        """Generate a VP; verb agrees with subject in number."""
        if rng.random() < P_TRANSITIVE:
            v_tok = rng.choice(VTR[lang][subj.number])
            emit(v_tok, 'v_agree',
                 target_number=subj.number,
                 subj_noun_pos=subj.noun_pos)
            sample_np(is_subject=False)
        else:
            v_tok = rng.choice(VINTR[lang][subj.number])
            emit(v_tok, 'v_agree',
                 target_number=subj.number,
                 subj_noun_pos=subj.noun_pos)

    def sample_pp():
        nonlocal pp_depth
        prep = rng.choice(PREPS[lang])
        emit(prep, 'structural')
        pp_depth += 1
        sample_np(is_subject=False)
        pp_depth -= 1

    def sample_rel(head: NPFeatures):
        nonlocal rel_depth
        rpro = RPROS[lang]
        emit(rpro, 'structural')
        rel_depth += 1
        # Verb in REL agrees with head noun's number
        if rng.random() < P_TRANSITIVE:
            v_tok = rng.choice(VTR[lang][head.number])
            emit(v_tok, 'v_agree_rel',
                 target_number=head.number,
                 head_noun_pos=head.noun_pos)
            sample_np(is_subject=False)
        else:
            v_tok = rng.choice(VINTR[lang][head.number])
            emit(v_tok, 'v_agree_rel',
                 target_number=head.number,
                 head_noun_pos=head.noun_pos)
        rel_depth -= 1

    def sample_clause():
        subj = sample_np(is_subject=True)
        sample_vp(subj)
        emit(DOT, 'structural')

    # ── Sentence-level ──
    sample_clause()
    if rng.random() < P_COMPOUND:
        emit(CONJS[lang], 'structural')
        sample_clause()

    return tokens, annots


def sample_packed_sequence(lang: str, rng: random.Random, max_len: int = MAX_LEN):
    """
    Pack multiple sentences into one training example.
    Returns (token_ids, annotations).
    Token IDs are padded to max_len.
    """
    tokens = [BOS, LANG_TOK[lang]]
    annots = [{'subcomp': 'bos'}, {'subcomp': 'lang_id'}]

    while len(tokens) < max_len - 1:
        sent_toks, sent_anns = sample_sentence(lang, rng)
        if len(tokens) + len(sent_toks) + 1 > max_len:  # +1 for EOS
            break
        tokens.extend(sent_toks)
        annots.extend(sent_anns)

    tokens.append(EOS)
    annots.append({'subcomp': 'eos'})

    # Pad
    pad_len = max_len - len(tokens)
    tokens.extend([PAD] * pad_len)
    annots.extend([{'subcomp': 'pad'}] * pad_len)

    ids = [STOI[t] for t in tokens]
    return ids, annots, lang


# ══════════════════════════════════════════════════════════════════════


In [4]:
# CELL 3.5 — Leakage confound audit for L_E (NEW — run before trusting
# any L_E number)
# ══════════════════════════════════════════════════════════════════════
#
# The claim we need to verify: for L_E, the upcoming noun's gender at
# the adj_agree_pre position is statistically independent of EVERYTHING
# generated so far in the sequence (subject, verb, earlier NPs, this
# NP's own DET/NUM). If any residual regularity exists, a bigger model
# improving on adj_agree_pre wouldn't be "ε resisting capacity as
# predicted" — it would be "found an unintended shortcut", producing a
# real capacity-dependent effect but not the one the theory predicts.
#
# By construction (gender = rng.choice(['M','F']) freshly at every
# sample_np call, independent of rng state consumed elsewhere) this
# should hold — but we check it empirically rather than by inspection
# alone, per the "diagnose before fixing / validate the instrument
# before scaling" principle.
#
# Method: generate many L_E examples, and for each adj_agree_pre
# position, test whether the target gender can be predicted above
# chance from (a) the immediately preceding tokens (DET_E, NUM_E — a
# sanity check that these really carry zero info), and (b) any
# upstream token in the same packed sequence (subject noun/verb/prior
# NP gender, if a compound sentence or object NP precedes). A simple,
# assumption-light way to do (b): for compound sentences, check whether
# the first clause's NP genders predict the second clause's NP gender
# at its own adj_agree_pre position.

def leakage_audit_L_E(n=20000, seed=13):
    rng = random.Random(seed)

    # (a) Local check: DET_E/NUM_E carry no gender info by construction
    #     (DET_E is a single invariant token — trivially uninformative).
    #     NUM_E is drawn independently of gender in the same sample_np
    #     call, so number should not predict gender either.
    num_given_gender = defaultdict(lambda: defaultdict(int))

    # (b) Cross-NP check: within one packed sequence, does any earlier
    #     NP's gender (subject, object, or first-clause NPs in a
    #     compound sentence) predict a later L_E adj_agree_pre gender?
    cross_np_pairs = []  # list of (earlier_gender, later_target_gender)

    n_adj_pre = 0

    for _ in range(n):
        ids, annots, _ = sample_packed_sequence('E', rng)
        seen_genders = []  # genders of NPs fully resolved so far (has noun)
        pending_target = None

        for i, a in enumerate(annots):
            sc = a['subcomp']
            if sc == 'det':
                continue
            if sc == 'num':
                # look back to find the most recent adj_agree_pre (if any)
                # is handled below; nothing to do here for the leakage
                # check itself except that num doesn't carry gender by
                # construction (checked via num_given_gender below).
                continue
            if sc == 'adj_agree_pre':
                n_adj_pre += 1
                tgt = a['target_gender']
                for g in seen_genders:
                    cross_np_pairs.append((g, tgt))
                pending_target = tgt
            if sc == 'n_after_det':
                seen_genders.append(a['gender'])
                # record NUM->gender co-occurrence: find the num token
                # immediately preceding this NP's det+num+[adj]+n span.
                # Cheap approximate check: look at annots[i-1] or [i-2].
                for back in (1, 2):
                    j = i - back
                    if j >= 0 and annots[j]['subcomp'] == 'num':
                        num_tok = ITOS[ids[j]]
                        num_given_gender[num_tok][a['gender']] += 1
                        break

    print(f"=== L_E leakage audit (n={n} sequences, {n_adj_pre} adj_agree_pre positions) ===")

    # (a) NUM -> gender association (should be ~50/50 for every NUM token)
    print("\n(a) NUM token vs. co-occurring noun gender (expect ~50/50 each):")
    max_dev = 0.0
    for num_tok, counts in sorted(num_given_gender.items()):
        total = sum(counts.values())
        m = counts.get('M', 0) / max(total, 1)
        f = counts.get('F', 0) / max(total, 1)
        dev = abs(m - 0.5)
        max_dev = max(max_dev, dev)
        print(f"    {num_tok}: M={m:.3f} F={f:.3f} (n={total})")
    print(f"    max deviation from 0.5: {max_dev:.4f}  "
          f"({'OK' if max_dev < 0.03 else 'INVESTIGATE'} — expect sampling noise only)")

    # (b) Earlier-NP-gender -> later-adj_agree_pre-target-gender
    print(f"\n(b) Earlier NP gender vs. later adj_agree_pre target gender "
          f"(n_pairs={len(cross_np_pairs)}):")
    if cross_np_pairs:
        same = sum(1 for g, t in cross_np_pairs if g == t)
        rate = same / len(cross_np_pairs)
        # chi-squared-ish sanity: rate should be ~0.5 (independent binary variables)
        se = math.sqrt(0.5 * 0.5 / len(cross_np_pairs))
        z = (rate - 0.5) / se
        print(f"    P(earlier gender == later target gender) = {rate:.4f}  "
              f"(expected 0.500, z={z:+.2f})")
        print(f"    {'OK' if abs(z) < 3 else 'INVESTIGATE — possible leakage'} "
              f"(|z| < 3 ~ consistent with independence)")
    else:
        print("    No cross-NP pairs observed in this sample (rare with n this "
              "size) — rerun with larger n if this persists.")

    ok = (max_dev < 0.03) and (not cross_np_pairs or abs(z) < 3)
    print(f"\n{'PASS' if ok else 'FAIL'}: "
          f"{'no leakage confound detected — safe to proceed.' if ok else 'possible leakage — diagnose before training any model on this.'}")
    return ok

leakage_ok = leakage_audit_L_E()
assert leakage_ok, "Leakage audit failed — fix the L_E sampler before proceeding."


# ══════════════════════════════════════════════════════════════════════


=== L_E leakage audit (n=20000 sequences, 193860 adj_agree_pre positions) ===

(a) NUM token vs. co-occurring noun gender (expect ~50/50 each):
    np_E: M=0.499 F=0.501 (n=241370)
    ns_E: M=0.501 F=0.499 (n=240874)
    max deviation from 0.5: 0.0014  (OK — expect sampling noise only)

(b) Earlier NP gender vs. later adj_agree_pre target gender (n_pairs=2267942):
    P(earlier gender == later target gender) = 0.4997  (expected 0.500, z=-0.78)
    OK (|z| < 3 ~ consistent with independence)

PASS: no leakage confound detected — safe to proceed.


In [5]:
# CELL 4 — Sampler sanity checks
# ══════════════════════════════════════════════════════════════════════

def sampler_check(n=2000, seed=42):
    rng = random.Random(seed)
    stats = {L: defaultdict(int) for L in LANGS}
    agreement_violations = {L: 0 for L in LANGS}
    total_adj = {L: 0 for L in LANGS}
    total_v = {L: 0 for L in LANGS}
    seq_lens = {L: [] for L in LANGS}

    for _ in range(n):
        lang = rng.choice(LANGS)
        ids, annots, _ = sample_packed_sequence(lang, rng)

        # Count non-pad tokens
        n_real = sum(1 for a in annots if a['subcomp'] != 'pad')
        seq_lens[lang].append(n_real)

        for a in annots:
            stats[lang][a['subcomp']] += 1

            if a['subcomp'] in ('adj_agree', 'adj_agree_pre'):
                total_adj[lang] += 1

            if a['subcomp'] in ('v_agree', 'v_agree_rel'):
                total_v[lang] += 1

    print("=== Sampler Statistics ===")
    for L in LANGS:
        print(f"\nL_{L}:")
        lens = seq_lens[L]
        print(f"  sequences: {len(lens)}")
        print(f"  avg tokens/seq: {np.mean(lens):.1f} (min {min(lens)}, max {max(lens)})")
        print(f"  sub-computation counts:")
        for sc, cnt in sorted(stats[L].items()):
            if sc not in ('pad', 'bos', 'eos'):
                print(f"    {sc}: {cnt}")
        print(f"  adj_agree(+pre) positions: {total_adj[L]}")
        print(f"  v_agree positions:         {total_v[L]}")

    # Spot-check: print 2 examples per language
    print("\n=== Sample sequences ===")
    rng2 = random.Random(0)
    for L in LANGS:
        print(f"\nL_{L}:")
        for _ in range(2):
            ids, annots, _ = sample_packed_sequence(L, rng2)
            toks = [ITOS[i] for i in ids if ITOS[i] != PAD]
            print(f"  {' '.join(toks)}")

            for i, a in enumerate(annots):
                if a['subcomp'] in ('adj_agree', 'adj_agree_pre', 'v_agree'):
                    print(f"    pos {i} [{ITOS[ids[i]]}]: {a}")

sampler_check()


# ══════════════════════════════════════════════════════════════════════


=== Sampler Statistics ===

L_A:
  sequences: 432
  avg tokens/seq: 118.3 (min 94, max 128)
  sub-computation counts:
    adj_agree: 4094
    det: 10335
    lang_id: 432
    n_after_det: 10335
    num: 10335
    structural: 9003
    v_agree: 4693
    v_agree_rel: 1010
  adj_agree(+pre) positions: 4094
  v_agree positions:         5703

L_B:
  sequences: 386
  avg tokens/seq: 121.3 (min 99, max 128)
  sub-computation counts:
    adj_agree: 4769
    det: 11977
    lang_id: 386
    n_after_det: 11977
    structural: 10364
    v_agree: 5450
    v_agree_rel: 1112
  adj_agree(+pre) positions: 4769
  v_agree positions:         6562

L_C:
  sequences: 379
  avg tokens/seq: 119.0 (min 91, max 128)
  sub-computation counts:
    adj_agree: 3636
    det: 9146
    lang_id: 379
    n_after_det: 9146
    num: 9146
    structural: 7900
    v_agree: 4134
    v_agree_rel: 840
  adj_agree(+pre) positions: 3636
  v_agree positions:         4974

L_D:
  sequences: 401
  avg tokens/seq: 119.3 (min 95, max 1

In [6]:
# CELL 5 — Verify agreement is correct
# ══════════════════════════════════════════════════════════════════════
# Rigorous check: for each adj_agree(_pre) position, verify the ADJ
# token's gender matches the annotation's target_gender. Same for V.

def verify_agreement(n=5000, seed=123):
    rng = random.Random(seed)
    n_adj_checked = 0
    n_adj_wrong = 0
    n_v_checked = 0
    n_v_wrong = 0

    for _ in range(n):
        lang = rng.choice(LANGS)
        ids, annots, _ = sample_packed_sequence(lang, rng)

        for i, a in enumerate(annots):
            if a['subcomp'] in ('adj_agree', 'adj_agree_pre'):
                tok = ITOS[ids[i]]
                target_g = a['target_gender']
                if tok not in ADJS[lang][target_g]:
                    n_adj_wrong += 1
                n_adj_checked += 1

            if a['subcomp'] in ('v_agree', 'v_agree_rel'):
                tok = ITOS[ids[i]]
                target_n = a['target_number']
                if tok not in VTR[lang][target_n] and tok not in VINTR[lang][target_n]:
                    n_v_wrong += 1
                n_v_checked += 1

    print(f"ADJ agreement: {n_adj_checked} checked, {n_adj_wrong} violations")
    print(f"V agreement:   {n_v_checked} checked, {n_v_wrong} violations")
    assert n_adj_wrong == 0, "ADJ agreement violations found!"
    assert n_v_wrong == 0, "V agreement violations found!"
    print("All agreement checks passed.")

verify_agreement()


# ══════════════════════════════════════════════════════════════════════


ADJ agreement: 51170 checked, 0 violations
V agreement:   71476 checked, 0 violations
All agreement checks passed.


In [7]:
# CELL 6 — Dataset + DataLoader
# ══════════════════════════════════════════════════════════════════════

class MultilingualPCFGDataset(IterableDataset):
    """
    Infinite stream of packed multilingual sequences.
    Languages sampled uniformly.
    Yields (x, y, lang_idx) where:
      x = input ids [:-1], y = target ids [1:]
      lang_idx = index into LANGS
    """
    def __init__(self, seed=0):
        self.seed = seed

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            seed = self.seed + worker_info.id * 100_000
        else:
            seed = self.seed
        rng = random.Random(seed)

        while True:
            lang = rng.choice(LANGS)
            ids, annots, _ = sample_packed_sequence(lang, rng)
            lang_idx = LANGS.index(lang)

            x = torch.tensor(ids[:-1], dtype=torch.long)
            y = torch.tensor(ids[1:],  dtype=torch.long)

            # Annotations are NOT serialized here — every training call
            # site discards them anyway; eval functions resample directly.
            yield x, y, lang_idx


def make_loader(seed=0, batch_size=256, num_workers=0, prefetch_factor=4):
    ds = MultilingualPCFGDataset(seed=seed)
    kwargs = dict(batch_size=batch_size, num_workers=num_workers,
                  pin_memory=(device == 'cuda'))
    if num_workers > 0:
        kwargs['prefetch_factor'] = prefetch_factor
        kwargs['persistent_workers'] = True
    return DataLoader(ds, **kwargs)


# ══════════════════════════════════════════════════════════════════════


In [8]:
# CELL 7 — Model (minimal GPT, same family as v5–v7)
# ══════════════════════════════════════════════════════════════════════

@dataclass
class GPTConfig:
    vocab_size: int = VOCAB_SIZE
    block_size: int = MAX_LEN - 1   # context = MAX_LEN - 1
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0
    bias: bool = False

SIZE_CONFIGS = {
    "10K":  GPTConfig(n_layer=2,  n_head=2,  n_embd=48),
    "25K":  GPTConfig(n_layer=2,  n_head=2,  n_embd=64),
    "50K":  GPTConfig(n_layer=2,  n_head=2,  n_embd=96),
    "100K": GPTConfig(n_layer=3,  n_head=4,  n_embd=96),
    "250K": GPTConfig(n_layer=4,  n_head=4,  n_embd=112),
    "500K": GPTConfig(n_layer=4,  n_head=4,  n_embd=128),
    "1M":   GPTConfig(n_layer=4,  n_head=4,  n_embd=160),
}


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.register_buffer("mask",
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                 .view(1, 1, cfg.block_size, cfg.block_size))
        self._last_head_out = None   # for activation extraction

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = att @ v   # (B, H, T, D)
        self._last_head_out = out.detach()
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(out)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self._last_neuron = None   # for activation extraction

    def forward(self, x):
        h = self.gelu(self.c_fc(x))
        self._last_neuron = h.detach()
        return self.c_proj(h)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight  # weight tying
        n_params = sum(p.numel() for p in self.parameters())
        print(f"GPT: {n_params:,} params  (L={cfg.n_layer} H={cfg.n_head} D={cfg.n_embd})")

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        return self.head(h)


# ══════════════════════════════════════════════════════════════════════


In [9]:
# CELL 8 — Training loop
# ══════════════════════════════════════════════════════════════════════

def train_model(cfg, max_steps=30000, eval_every=1000, seed=0,
                batch_size=256, lr=3e-3, log=True, num_workers=0,
                patience=5):
    """
    Train a GPT on the multilingual PCFG LM task.
    Returns (model, log_dict).
    Convergence: early-stop if val loss doesn't improve for `patience` evals.
    """
    torch.manual_seed(seed)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loader = make_loader(seed=seed, batch_size=batch_size, num_workers=num_workers)
    it = iter(loader)

    pad_id = STOI[PAD]
    log_dict = {'steps': [], 'train_loss': [], 'val_loss': []}
    best_val = float('inf')
    no_improve = 0
    t0 = time.time()

    for step in range(1, max_steps + 1):
        model.train()
        x, y, lang_idx = next(it)
        x, y = x.to(device), y.to(device)
        logits = model(x)
        mask = (y != pad_id)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1),
                               reduction='none')
        loss = (loss * mask.view(-1).float()).sum() / mask.sum()
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % eval_every == 0 or step == max_steps:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for _ in range(20):
                    x, y, _ = next(it)
                    x, y = x.to(device), y.to(device)
                    logits = model(x)
                    mask = (y != pad_id)
                    vl = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1),
                                        reduction='none')
                    vl = (vl * mask.view(-1).float()).sum() / mask.sum()
                    val_losses.append(vl.item())
            val = np.mean(val_losses)
            log_dict['steps'].append(step)
            log_dict['train_loss'].append(loss.item())
            log_dict['val_loss'].append(val)
            if log:
                print(f"  step {step:5d}: train {loss.item():.4f}  "
                      f"val {val:.4f}  ({time.time()-t0:.1f}s)")
            if val < best_val - 1e-4:
                best_val = val
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= patience and step >= eval_every * 3:
                if log:
                    print(f"  [converged at step {step}]")
                break

    log_dict['final_val'] = best_val
    return model, log_dict


# ══════════════════════════════════════════════════════════════════════


In [11]:
# CELL 8.5 — Worker count benchmark (run once, on your actual machine)
# ══════════════════════════════════════════════════════════════════════
# Not needed yet — only matters once you're actually training at scale.
# Left commented, same as v7.

def benchmark_num_workers(worker_counts=(0, 4, 8, 16, 24, 32),
                          n_steps=100, cfg=None):
    if cfg is None:
        cfg = SIZE_CONFIGS["10K"]
    results = {}
    for nw in worker_counts:
        torch.manual_seed(0)
        model = GPT(cfg).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
        loader = make_loader(seed=0, batch_size=256, num_workers=nw)
        it = iter(loader)
        pad_id = STOI[PAD]

        for _ in range(5):
            x, y, _ = next(it)
        if device == 'cuda':
            torch.cuda.synchronize()

        t0 = time.time()
        for _ in range(n_steps):
            x, y, _ = next(it)
            x, y = x.to(device), y.to(device)
            logits = model(x)
            mask = (y != pad_id)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1),
                                   reduction='none')
            loss = (loss * mask.view(-1).float()).sum() / mask.sum()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if device == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - t0

        del loader, it, model, opt
        results[nw] = elapsed
        print(f"  num_workers={nw:>3d}: {elapsed:.2f}s for {n_steps} steps "
              f"({elapsed/n_steps*1000:.1f} ms/step)")

    best = min(results, key=results.get)
    print(f"\nFastest: num_workers={best} ({results[best]/n_steps*1000:.1f} ms/step)")
    return results

# Uncomment to run (takes a few minutes total across all worker counts):
# worker_bench = benchmark_num_workers()


# ══════════════════════════════════════════════════════════════════════


In [10]:
# CELL 8.7 — Wilson confidence intervals (NEW — Lesson 4)
# ══════════════════════════════════════════════════════════════════════

def wilson_ci(correct, total, z=1.96):
    """Wilson score interval for a binomial proportion. Returns (lo, hi)."""
    if total == 0:
        return (0.0, 1.0)
    phat = correct / total
    denom = 1 + z * z / total
    center = phat + z * z / (2 * total)
    margin = z * math.sqrt(phat * (1 - phat) / total + z * z / (4 * total * total))
    lo = (center - margin) / denom
    hi = (center + margin) / denom
    return (max(0.0, lo), min(1.0, hi))


# ══════════════════════════════════════════════════════════════════════


In [13]:
# CELL 9 — Per-sub-computation accuracy evaluation
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def eval_subcomp_accuracy(model, n=512, seed=777, min_n_for_trust=100):
    """
    For each (language, sub-computation), compute top-1 accuracy
    at the annotated positions, with Wilson CIs.

    Key sub-comps:
      adj_agree      — predict ADJ form, post-nominal (can copy from N)
      adj_agree_pre  — predict ADJ form, pre-nominal (D, E only) [ε test]
      v_agree        — predict V form (needs number)    [ε control]
      n_after_det    — predict N (constrained by DET?)  [τ test]
      structural     — predict DOT/CONJ/PREP            [ε≈0 control]
    """
    model.eval()
    rng = random.Random(seed)

    counts = defaultdict(lambda: [0, 0])

    for _ in range(n):
        lang = rng.choice(LANGS)
        ids, annots, _ = sample_packed_sequence(lang, rng)
        x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
        logits = model(x)  # (1, T, V)

        for i, ann in enumerate(annots[1:], start=1):
            sc = ann['subcomp']
            if sc in ('pad', 'bos', 'eos', 'lang_id'):
                continue
            pred = logits[0, i - 1].argmax().item()
            correct = (pred == ids[i])
            counts[(lang, sc)][0] += int(correct)
            counts[(lang, sc)][1] += 1

    print("=== Per-sub-computation accuracy (with Wilson 95% CI) ===")
    subcmps = sorted(set(sc for _, sc in counts.keys()))
    for sc in subcmps:
        print(f"\n  {sc}:")
        for L in LANGS:
            c, t = counts.get((L, sc), (0, 0))
            if t == 0:
                continue
            acc = c / t
            lo, hi = wilson_ci(c, t)
            flag = "" if t >= min_n_for_trust else "  [LOW N — do not trust]"
            print(f"    L_{L}: {acc:.3f}  [{lo:.3f}, {hi:.3f}]  (n={t}){flag}")
    return dict(counts)


# ══════════════════════════════════════════════════════════════════════


In [24]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 — Calibration pilot (corrected gate)
# ══════════════════════════════════════════════════════════════════════
# Train a 10K model and check the L_E gate before spending compute on
# the full sweep.
#
# NOTE: full-vocab top-1 accuracy (eval_subcomp_accuracy) is NOT the
# right signal to gate on here. With N_ADJ_LEMMAS=8, lemma choice
# within the correct gender is uniform noise no model can predict, so
# even PERFECT gender reading caps full-vocab accuracy at ~1/8=0.125,
# and NO gender signal caps it at ~1/16=0.0625 — not 1.0 / 0.5. Use
# eval_feature_accuracy (restricted to the 16 real candidates, scores
# gender directly) for the actual gate: chance=0.500, ceiling=1.0.

print("=== Calibration pilot: 10K model ===")
pilot_model, pilot_log = train_model(
    SIZE_CONFIGS["10K"], max_steps=40000, eval_every=2000, seed=0,
    num_workers=NUM_WORKERS
)
pilot_acc = eval_subcomp_accuracy(pilot_model, n=1000, seed=777)

print("\n=== L_E ε-gate check (feature-level — the real signal) ===")
pilot_feat_acc = eval_feature_accuracy(pilot_model, n=1000, seed=777)

d_c, d_t = pilot_feat_acc.get(('D', 'adj_agree_pre'), (0, 1))
e_c, e_t = pilot_feat_acc.get(('E', 'adj_agree_pre'), (0, 1))
d_acc, d_ci = d_c / max(d_t, 1), wilson_ci(d_c, d_t)
e_acc, e_ci = e_c / max(e_t, 1), wilson_ci(e_c, e_t)

print(f"L_D adj_agree_pre (feature-level): {d_acc:.3f}  {d_ci}  (n={d_t})  "
      f"— expect near 1.000 (ceiling)")
print(f"L_E adj_agree_pre (feature-level): {e_acc:.3f}  {e_ci}  (n={e_t})  "
      f"— expect near 0.500 (chance)")

if e_ci[0] > 0.55:
    print("  GATE FAILED: L_E's CI excludes chance already at 10K — "
          "re-check leakage audit with larger n before proceeding.")
elif d_ci[1] < 0.90:
    print("  GATE FAILED: L_D isn't near ceiling — the DET_D gender "
          "signal isn't being read at the ADJ position; check P_ADJ "
          "frequency and whether capacity is too low to afford the "
          "DET_D→ADJ pathway yet.")
else:
    print("  Gate passed — proceed to the full sweep (Cell 13).")

=== Calibration pilot: 10K model ===
GPT: 79,008 params  (L=2 H=2 D=48)


  step  2000: train 1.7507  val 1.7501  (32.1s)
  step  4000: train 1.7209  val 1.7033  (64.3s)
  step  6000: train 1.7039  val 1.6973  (96.1s)
  step  8000: train 1.6844  val 1.6888  (127.0s)
  step 10000: train 1.7029  val 1.6887  (158.7s)
  step 12000: train 1.6920  val 1.6930  (190.5s)
  step 14000: train 1.6814  val 1.6918  (222.6s)
  step 16000: train 1.6796  val 1.6913  (255.3s)
  step 18000: train 1.6920  val 1.6872  (287.1s)
  step 20000: train 1.6857  val 1.6910  (317.9s)
  step 22000: train 1.6714  val 1.6952  (349.9s)
  step 24000: train 1.7041  val 1.6960  (382.5s)
  step 26000: train 1.6832  val 1.6893  (413.8s)
  step 28000: train 1.6613  val 1.6920  (446.2s)
  [converged at step 28000]
=== Per-sub-computation accuracy (with Wilson 95% CI) ===

  adj_agree:
    L_A: 0.002  [0.001, 0.005]  (n=1976)
    L_B: 0.002  [0.001, 0.004]  (n=2492)
    L_C: 0.002  [0.001, 0.005]  (n=1871)

  adj_agree_pre:
    L_D: 0.115  [0.102, 0.130]  (n=2035)
    L_E: 0.051  [0.042, 0.062]  (n=

In [11]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8.8 — Checkpointing utilities (NEW)
# ══════════════════════════════════════════════════════════════════════

import os
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def ckpt_path(size_label, seed):
    return os.path.join(CKPT_DIR, f"model_{size_label}_seed{seed}.pt")

def save_checkpoint(model, cfg, log, size_label, seed):
    path = ckpt_path(size_label, seed)
    torch.save({
        'state_dict': model.state_dict(),
        'cfg': cfg,          # GPTConfig dataclass — pickles fine
        'log': log,
        'size_label': size_label,
        'seed': seed,
    }, path)
    print(f"  [saved checkpoint: {path}]")
    return path

def load_checkpoint(size_label, seed, map_location=None):
    path = ckpt_path(size_label, seed)
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=map_location or device)
    model = GPT(ckpt['cfg']).to(device)
    model.load_state_dict(ckpt['state_dict'])
    return model, ckpt['log']


# ── Save the pilot model you already trained, right now ──
#save_checkpoint(pilot_model, SIZE_CONFIGS["10K"], pilot_log, "10K", seed=0)

In [14]:
# CELL 10.5 — Restricted-vocab feature accuracy
# ══════════════════════════════════════════════════════════════════════
# Full-vocab top-1 accuracy is unreliable at branching positions (a
# single competing token, like RPRO, can dominate the argmax) and
# conflates "got the feature right" with "guessed the right lemma"
# (lemma choice is uniform random noise by design — unlearnable).
# This restricts argmax to the position's real candidate set and scores
# the FEATURE (gender / number) of the predicted token, not the exact
# lemma. Chance floor = 0.500.

ADJ_GENDER: Dict[str, str] = {}
for L in LANGS:
    for g in ['M', 'F']:
        for tok in ADJS[L][g]:
            ADJ_GENDER[tok] = g

VERB_NUMBER: Dict[str, str] = {}
for L in LANGS:
    for n in ['SG', 'PL']:
        for tok in VTR[L][n]:
            VERB_NUMBER[tok] = n
        for tok in VINTR[L][n]:
            VERB_NUMBER[tok] = n


@torch.no_grad()
def eval_feature_accuracy(model, n=1000, seed=777):
    """
    Feature-level accuracy for adj_agree, adj_agree_pre, v_agree,
    v_agree_rel: restrict argmax to the position's valid vocab subset,
    then check whether the predicted token's FEATURE (gender/number)
    matches the target feature. Chance = 0.500. Reports Wilson CIs.
    """
    model.eval()
    rng = random.Random(seed)

    adj_ids = {L: torch.tensor([STOI[t] for t in ADJS[L]['M'] + ADJS[L]['F']],
                               device=device) for L in LANGS}
    verb_ids = {L: torch.tensor(
        [STOI[t] for t in VTR[L]['SG'] + VTR[L]['PL'] +
         VINTR[L]['SG'] + VINTR[L]['PL']], device=device) for L in LANGS}

    counts = defaultdict(lambda: [0, 0])

    for _ in range(n):
        lang = rng.choice(LANGS)
        ids, annots, _ = sample_packed_sequence(lang, rng)
        x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
        logits = model(x)[0]  # (T, V)

        for i, ann in enumerate(annots[1:], start=1):
            sc = ann['subcomp']
            if sc in ('adj_agree', 'adj_agree_pre'):
                cand, target_feat, feat_lookup = adj_ids[lang], ann['target_gender'], ADJ_GENDER
            elif sc in ('v_agree', 'v_agree_rel'):
                cand, target_feat, feat_lookup = verb_ids[lang], ann['target_number'], VERB_NUMBER
            else:
                continue

            local_logits = logits[i - 1, cand]
            pred_tok = ITOS[cand[local_logits.argmax()].item()]
            pred_feat = feat_lookup[pred_tok]

            counts[(lang, sc)][0] += int(pred_feat == target_feat)
            counts[(lang, sc)][1] += 1

    print("=== Restricted-vocab feature accuracy (chance = 0.500), with Wilson CI ===")
    subcmps = sorted(set(sc for _, sc in counts.keys()))
    for sc in subcmps:
        print(f"\n  {sc}:")
        for L in LANGS:
            c, t = counts.get((L, sc), (0, 0))
            if t == 0:
                continue
            acc = c / t
            lo, hi = wilson_ci(c, t)
            print(f"    L_{L}: {acc:.3f}  [{lo:.3f}, {hi:.3f}]  (n={t})")
    return dict(counts)


# print("\n=== D vs E pre-nominal ADJ ε-test (feature-level) ===")
# pilot_feat_acc = eval_feature_accuracy(pilot_model, n=1000, seed=777)
# d_c, d_t = pilot_feat_acc.get(('D', 'adj_agree_pre'), (0, 1))
# e_c, e_t = pilot_feat_acc.get(('E', 'adj_agree_pre'), (0, 1))
# print(f"L_D adj_agree_pre: {d_c/max(d_t,1):.3f}  {wilson_ci(d_c, d_t)}  (n={d_t})")
# print(f"L_E adj_agree_pre: {e_c/max(e_t,1):.3f}  {wilson_ci(e_c, e_t)}  (n={e_t})")
# print("  → L_D near ceiling + L_E near 0.500 is the expected pattern:")
# print("    L_D's DET marks gender (cheap to read before the noun exists);")
# print("    L_E's DET is invariant, so gender genuinely can't be committed")
# print("    to at this position — the position the theory needs to exist.")


# ══════════════════════════════════════════════════════════════════════


In [16]:
# CELL 10.6 — Structural breakdown (restricted-vocab, per token type)
# ══════════════════════════════════════════════════════════════════════
# `structural` lumps DOT/CONJ/PREP/RPRO together under full-vocab argmax.
# Two of those are branching-contaminated the same way adj_agree was:
#   - conj: competes with "start of new NP" (surface forms depend on
#     language) after DOT. Languages with an invariant determiner
#     (C, E) have only ONE determiner surface form, so DET alone can
#     out-mass CONJ on full-vocab argmax — a determiner-inventory
#     artifact, unrelated to ε.
#   - prep: splits its branch probability across N_PREPS forms, diluting
#     any single prep token's mass everywhere, in all languages equally.
# Fix: restrict argmax to the real local competitor set at each slot.

@torch.no_grad()
def eval_structural_breakdown(model, n=1000, seed=777):
    model.eval()
    rng = random.Random(seed)
    counts = defaultdict(lambda: [0, 0])          # full-vocab (for reference)
    counts_fixed = defaultdict(lambda: [0, 0])     # restricted-vocab (trustworthy)

    # Per-language competitor sets for the post-DOT branch: CONJ vs all
    # DET surface forms for that language (i.e. "start of new sentence").
    det_surface = {
        'A': list(DETS_A.values()),
        'B': list(DETS_B.values()),
        'C': [DET_C],
        'D': list(DETS_D.values()),
        'E': [DET_E],
    }
    post_dot_ids = {
        L: torch.tensor([STOI[CONJS[L]]] + [STOI[t] for t in det_surface[L]],
                        device=device)
        for L in LANGS
    }
    conj_id = {L: STOI[CONJS[L]] for L in LANGS}

    prep_ids = {L: torch.tensor([STOI[t] for t in PREPS[L]], device=device)
                for L in LANGS}

    def tok_type(lang, tok):
        if tok == DOT: return 'dot'
        if tok == CONJS[lang]: return 'conj'
        if tok in PREPS[lang]: return 'prep'
        if tok == RPROS[lang]: return 'rpro'
        return 'other'

    for _ in range(n):
        lang = rng.choice(LANGS)
        ids, annots, _ = sample_packed_sequence(lang, rng)
        x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
        logits = model(x)[0]

        for i, ann in enumerate(annots[1:], start=1):
            if ann['subcomp'] != 'structural':
                continue
            tt = tok_type(lang, ITOS[ids[i]])

            pred = logits[i - 1].argmax().item()
            counts[(lang, tt)][0] += int(pred == ids[i])
            counts[(lang, tt)][1] += 1

            if tt == 'conj':
                cand = post_dot_ids[lang]
                local_pred = cand[logits[i - 1, cand].argmax()].item()
                correct = int(local_pred == conj_id[lang])
                counts_fixed[(lang, tt)][0] += correct
                counts_fixed[(lang, tt)][1] += 1
            elif tt == 'prep':
                cand = prep_ids[lang]
                local_pred = cand[logits[i - 1, cand].argmax()].item()
                correct = int(local_pred == ids[i])
                counts_fixed[(lang, tt)][0] += correct
                counts_fixed[(lang, tt)][1] += 1
            else:
                counts_fixed[(lang, tt)][0] += int(pred == ids[i])
                counts_fixed[(lang, tt)][1] += 1

    def _print_table(title, cdict):
        print(title)
        types = sorted(set(t for _, t in cdict.keys()))
        header = f"{'type':<8}" + "".join(f"{'L_'+L:>10}" for L in LANGS)
        print(header)
        for tt in types:
            row = f"{tt:<8}"
            for L in LANGS:
                c, t = cdict.get((L, tt), (0, 0))
                acc = c / max(t, 1)
                row += f"{acc:>9.3f} " if t > 0 else f"{'N/A':>10}"
            print(row)

    _print_table("=== structural accuracy — full-vocab (original, unreliable) ===", counts)
    print()
    _print_table("=== structural accuracy — restricted-vocab (trustworthy) ===", counts_fixed)
    return dict(counts), dict(counts_fixed)


# pilot_structural, pilot_structural_fixed = eval_structural_breakdown(pilot_model, n=1000, seed=777)
#
# # Clean up
# del pilot_model
# if device == 'cuda':
#     torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════


In [25]:
# CELL 11 — Activation extraction + sharing scores
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def collect_activation_profile(model, lang, subcomp_filter,
                               n_examples=512, seed=999):
    """
    Collect mean activation magnitude per head + per neuron
    at positions matching `subcomp_filter` for language `lang`.
    Returns a 1-D numpy vector (the "profile"), or None if that
    (lang, subcomp) combination has no positions (e.g. A/adj_agree_pre).
    """
    model.eval()
    rng = random.Random(seed)

    all_ids = []
    all_positions = []

    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp_filter and i < MAX_LEN - 1]
        if not positions:
            continue
        all_ids.append(ids[:-1])
        all_positions.append(positions)

    if not all_ids:
        return None

    batch_size = 64
    profile_parts = []

    for batch_start in range(0, len(all_ids), batch_size):
        batch_ids = all_ids[batch_start:batch_start + batch_size]
        batch_pos = all_positions[batch_start:batch_start + batch_size]
        x = torch.tensor(batch_ids, dtype=torch.long, device=device)
        _ = model(x)

        B_actual = x.shape[0]
        parts = []

        for blk in model.blocks:
            head_out = blk.attn._last_head_out  # (B, H, T, D_h)
            H = head_out.shape[1]
            D_h = head_out.shape[3]

            head_mags = []
            for b_idx in range(B_actual):
                pos = batch_pos[b_idx]
                if pos:
                    h = head_out[b_idx, :, pos, :]
                    head_mags.append(h.abs().mean(dim=(1, 2)))
            if head_mags:
                parts.append(torch.stack(head_mags).mean(dim=0).cpu().numpy())

            neuron = blk.mlp._last_neuron  # (B, T, D_mlp)
            D_m = neuron.shape[2]
            neuron_mags = []
            for b_idx in range(B_actual):
                pos = batch_pos[b_idx]
                if pos:
                    n = neuron[b_idx, pos, :]
                    neuron_mags.append(n.abs().mean(dim=0))
            if neuron_mags:
                parts.append(torch.stack(neuron_mags).mean(dim=0).cpu().numpy())

        if parts:
            profile_parts.append(np.concatenate(parts))

    if not profile_parts:
        return None
    return np.mean(profile_parts, axis=0)


def pearson(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a**2).sum() * (b**2).sum())
    if denom < 1e-12:
        return 0.0
    return float((a * b).sum() / denom)


def sharing_scores_by_subcomp(model, subcomps=None, n_examples=512, seed=999,
                              langs=None):
    """
    Compute pairwise Pearson sharing scores for each sub-computation.
    Returns {subcomp: {pair: score}} where pair is e.g. 'AB' or 'DE'.

    Pairs are generated from ALL combinations of `langs` (default:
    LANGS). Missing profiles (e.g. 'A' has no adj_agree_pre positions)
    are skipped automatically — this is how DE ends up being the only
    populated pair for adj_agree_pre.
    """
    if subcomps is None:
        subcomps = ['adj_agree', 'adj_agree_pre', 'v_agree', 'n_after_det', 'structural']
    if langs is None:
        langs = LANGS

    pairs = list(itertools.combinations(langs, 2))
    results = {}

    for sc in subcomps:
        profiles = {}
        for L in langs:
            p = collect_activation_profile(model, L, sc,
                                          n_examples=n_examples, seed=seed)
            if p is not None:
                profiles[L] = p

        scores = {}
        for a, b in pairs:
            if a in profiles and b in profiles:
                scores[f"{a}{b}"] = pearson(profiles[a], profiles[b])
            else:
                scores[f"{a}{b}"] = None
        results[sc] = scores

    return results


# ══════════════════════════════════════════════════════════════════════


In [17]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.5 — D vs E linear separability probe at adj_agree_pre,
# with a generic-convergence control (NEW)
# ══════════════════════════════════════════════════════════════════════
# Question: are D and E using dissociated circuits at adj_agree_pre, or
# does the raw Pearson-magnitude rise just reflect generic D-E
# representational convergence at scale (same slot-detection reflex,
# unrelated to gender)?
#
# A probe on adj_agree_pre ALONE can't distinguish these — if D and E
# converge everywhere with scale, a rising probe score there would be
# just as ambiguous as the rising Pearson score was. So: run the SAME
# probe on a control subcomp that has nothing to do with gender
# (structural — DOT/CONJ/PREP/RPRO), at the same two sizes. Only trust
# an adj_agree_pre trend that isn't also present in the control.

@torch.no_grad()
def collect_raw_activations(model, lang, subcomp_filter, n_examples=512, seed=999):
    """Per-position residual-stream activation (post-ln_f) at matching
    positions. Returns array of shape (n_positions, n_embd)."""
    model.eval()
    rng = random.Random(seed)
    all_ids, all_positions = [], []

    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp_filter and i < MAX_LEN - 1]
        if not positions:
            continue
        all_ids.append(ids[:-1])
        all_positions.append(positions)

    if not all_ids:
        return None

    reps = []
    batch_size = 64
    for start in range(0, len(all_ids), batch_size):
        batch_ids = all_ids[start:start + batch_size]
        batch_pos = all_positions[start:start + batch_size]
        x = torch.tensor(batch_ids, dtype=torch.long, device=device)

        pos = torch.arange(x.shape[1], device=device)
        h = model.tok_emb(x) + model.pos_emb(pos)
        for blk in model.blocks:
            h = blk(h)
        h = model.ln_f(h)  # (B, T, D)

        for b_idx, positions in enumerate(batch_pos):
            for p in positions:
                reps.append(h[b_idx, p].cpu().numpy())

    return np.array(reps) if reps else None


def dissociation_probe(model, subcomp, langs=('D', 'E'),
                       n_examples=512, seed=999, n_splits=5):
    """
    Cross-validated logistic-regression probe classifying which
    language an activation came from, at the given subcomp.
    ~0.5 = shared/indistinguishable representation.
    ~1.0 = fully dissociated (language-identifiable) representation.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    reps_by_lang = {}
    for L in langs:
        r = collect_raw_activations(model, L, subcomp, n_examples, seed)
        if r is None:
            print(f"    no {subcomp} positions for L_{L}")
            return None
        reps_by_lang[L] = r

    X = np.concatenate([reps_by_lang[L] for L in langs], axis=0)
    y = np.concatenate([[i] * len(reps_by_lang[L]) for i, L in enumerate(langs)])

    clf = LogisticRegression(max_iter=1000)
    scores = cross_val_score(clf, X, y, cv=n_splits)
    return float(scores.mean()), float(scores.std())


def dissociation_with_control(sizes=("10K", "1M"), seeds=(0, 1),
                              target_subcomp='adj_agree_pre',
                              control_subcomp='structural'):
    """
    Runs the dissociation probe on target_subcomp AND control_subcomp
    at each (size, seed), and reports whether target's trend across
    scale EXCEEDS control's trend. Only a target delta that clears the
    control delta counts as evidence of ε-specific dissociation rather
    than generic D-E convergence.
    """
    results = defaultdict(dict)  # (subcomp) -> {(size, seed): (mean, std)}

    for size in sizes:
        for seed in seeds:
            model, _ = load_checkpoint(size, seed)
            print(f"\n  {size} / seed {seed}:")

            t = dissociation_probe(model, subcomp=target_subcomp)
            if t is not None:
                print(f"    {target_subcomp:<15} (target):  {t[0]:.3f} ± {t[1]:.3f}")
                results[target_subcomp][(size, seed)] = t

            c = dissociation_probe(model, subcomp=control_subcomp)
            if c is not None:
                print(f"    {control_subcomp:<15} (control): {c[0]:.3f} ± {c[1]:.3f}")
                results[control_subcomp][(size, seed)] = c

            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    print(f"\n=== {target_subcomp} vs {control_subcomp}: scale trend comparison ===")
    for seed in seeds:
        t10 = results[target_subcomp].get((sizes[0], seed))
        t1m = results[target_subcomp].get((sizes[1], seed))
        c10 = results[control_subcomp].get((sizes[0], seed))
        c1m = results[control_subcomp].get((sizes[1], seed))
        if not all([t10, t1m, c10, c1m]):
            print(f"  seed {seed}: missing data, skipping")
            continue
        t_delta = t1m[0] - t10[0]
        c_delta = c1m[0] - c10[0]
        specific = t_delta - c_delta
        print(f"  seed {seed}:")
        print(f"    {target_subcomp} delta ({sizes[0]}->{sizes[1]}):  {t_delta:+.3f}")
        print(f"    {control_subcomp} delta ({sizes[0]}->{sizes[1]}): {c_delta:+.3f}")
        print(f"    ε-specific component (target - control): {specific:+.3f}  "
              f"{'[real, subcomp-specific trend]' if abs(specific) > 0.05 else '[not distinguishable from generic convergence]'}")

    return results


dissociation_results = dissociation_with_control()

GPT: 79,008 params  (L=2 H=2 D=48)

  10K / seed 0:
    adj_agree_pre   (target):  1.000 ± 0.000
    structural      (control): 1.000 ± 0.000
GPT: 79,008 params  (L=2 H=2 D=48)

  10K / seed 1:
    adj_agree_pre   (target):  1.000 ± 0.000
    structural      (control): 1.000 ± 0.000
GPT: 1,309,120 params  (L=4 H=4 D=160)

  1M / seed 0:
    adj_agree_pre   (target):  1.000 ± 0.000
    structural      (control): 1.000 ± 0.000
GPT: 1,309,120 params  (L=4 H=4 D=160)

  1M / seed 1:
    adj_agree_pre   (target):  1.000 ± 0.000
    structural      (control): 1.000 ± 0.000

=== adj_agree_pre vs structural: scale trend comparison ===
  seed 0:
    adj_agree_pre delta (10K->1M):  +0.000
    structural delta (10K->1M): +0.000
    ε-specific component (target - control): +0.000  [not distinguishable from generic convergence]
  seed 1:
    adj_agree_pre delta (10K->1M):  +0.000
    structural delta (10K->1M): +0.000
    ε-specific component (target - control): +0.000  [not distinguishable from ge

In [15]:
# CELL 12 — Pearson_same noise floor
# ══════════════════════════════════════════════════════════════════════

def pearson_same_noise_floor(cfg, lang='A', subcomp='adj_agree', n_seeds=3,
                             max_steps=30000, num_workers=0):
    """
    Cross-seed same-language sharing as noise floor.
    Train n_seeds models, compute Pearson(lang_seed_i, lang_seed_j) for
    the target sub-computation. Use lang='D' or 'E' when checking the
    noise floor for adj_agree_pre.
    """
    models = []
    for s in range(n_seeds):
        print(f"  training seed {s}...")
        m, _ = train_model(cfg, max_steps=max_steps, eval_every=2000,
                           seed=s, log=False, num_workers=num_workers)
        models.append(m)

    profiles = []
    for m in models:
        p = collect_activation_profile(m, lang, subcomp, n_examples=512)
        if p is not None:
            profiles.append(p)
        del m
    if device == 'cuda':
        torch.cuda.empty_cache()

    scores = []
    for i in range(len(profiles)):
        for j in range(i + 1, len(profiles)):
            scores.append(pearson(profiles[i], profiles[j]))

    print(f"  Pearson_same (L_{lang}, {subcomp}): {np.mean(scores):.3f} ± {np.std(scores):.3f}")
    return np.mean(scores)

# Not run automatically — compute-heavy (trains n_seeds models). Call
# manually once the calibration gate passes, e.g.:
# noise_floor_DE = pearson_same_noise_floor(SIZE_CONFIGS["10K"], lang='D',
#                                            subcomp='adj_agree_pre', n_seeds=3,
#                                            num_workers=NUM_WORKERS)


# ══════════════════════════════════════════════════════════════════════


In [26]:
noise_floor_DE = pearson_same_noise_floor(SIZE_CONFIGS["10K"], lang='D',
                                           subcomp='adj_agree_pre', n_seeds=3,
                                           num_workers=NUM_WORKERS)

  training seed 0...
GPT: 79,008 params  (L=2 H=2 D=48)
  training seed 1...
GPT: 79,008 params  (L=2 H=2 D=48)
  training seed 2...
GPT: 79,008 params  (L=2 H=2 D=48)
  Pearson_same (L_D, adj_agree_pre): 0.051 ± 0.075


In [35]:
noise_floor_D_1M = pearson_same_noise_floor(SIZE_CONFIGS["1M"], lang='D',
                                             subcomp='adj_agree_pre', n_seeds=3,
                                             num_workers=NUM_WORKERS)

  training seed 0...
GPT: 1,309,120 params  (L=4 H=4 D=160)
  training seed 1...
GPT: 1,309,120 params  (L=4 H=4 D=160)
  training seed 2...
GPT: 1,309,120 params  (L=4 H=4 D=160)
  Pearson_same (L_D, adj_agree_pre): 0.064 ± 0.061


In [32]:
print(type(models) if 'models' in dir() else "models: NOT FOUND")
if 'models' in dir():
    print(len(models), [type(m) for m in models])

models: NOT FOUND


In [12]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8.8 (patched again) — weights_only=False for self-generated checkpoints
# ══════════════════════════════════════════════════════════════════════

def load_checkpoint(size_label, seed, map_location=None):
    path = ckpt_path(size_label, seed)
    if not os.path.exists(path):
        return None
    # weights_only=False is safe here specifically because these
    # checkpoints were written by save_checkpoint() in this same
    # session/machine, not downloaded from anywhere. Don't use this
    # for checkpoints from an untrusted source.
    ckpt = torch.load(path, map_location=map_location or device, weights_only=False)
    model = GPT(ckpt['cfg']).to(device)
    model.load_state_dict(ckpt['state_dict'])
    return model, ckpt['log']

In [33]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13 (patched again) — run_sweep with selectable sizes
# ══════════════════════════════════════════════════════════════════════

def run_sweep(seeds=(0, 1), num_workers=0, resume=True, sizes=None):
    """
    Multi-seed sweep with checkpoint/resume, over `sizes` (default: the
    full SWEEP_SIZES list; pass sizes=("10K","1M") for a two-point run).
    """
    if sizes is None:
        sizes = SWEEP_SIZES

    results = {size: {} for size in sizes}
    for size_label in sizes:
        print(f"\n{'='*60}")
        print(f"  Size: {size_label}")
        print(f"{'='*60}")
        cfg = SIZE_CONFIGS[size_label]

        for seed in seeds:
            print(f"\n  -- seed {seed} --")

            loaded = load_checkpoint(size_label, seed) if resume else None
            if loaded is not None:
                model, log = loaded
                print(f"  [resumed from checkpoint, skipped training]")
            else:
                model, log = train_model(
                    cfg, max_steps=SWEEP_STEPS[size_label],
                    eval_every=2000, seed=seed, num_workers=num_workers
                )
                save_checkpoint(model, cfg, log, size_label, seed)

            acc = eval_subcomp_accuracy(model, n=800, seed=777 + seed)
            sharing = sharing_scores_by_subcomp(
                model, subcomps=SUBCOMPS_TO_MEASURE, n_examples=512, seed=999 + seed
            )

            results[size_label][seed] = {
                'accuracy': dict(acc),
                'sharing': sharing,
                'val_loss': log['final_val'],
            }

            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    return results


# ══════════════════════════════════════════════════════════════════════
# CELL 13.5 — Two-point (10K vs 1M) stability check (NEW)
# ══════════════════════════════════════════════════════════════════════
# Mirrors the v7 L_D accuracy-stability check (10K vs 1M with p-values),
# applied to the two things that actually matter for the L_E result:
#   (1) L_E's OWN adj_agree_pre feature accuracy — should stay flat
#       (p NOT significant) — A2 predicts ε doesn't shrink with capacity.
#   (2) The D-E Pearson sharing score on adj_agree_pre — this is what
#       SHOULD move if N* for this pair falls inside [10K, 1M].

def two_proportion_z(c1, t1, c2, t2):
    """Two-proportion z-test (pooled). Returns (z, two_tailed_p)."""
    if t1 == 0 or t2 == 0:
        return None, None
    p1, p2 = c1 / t1, c2 / t2
    p_pool = (c1 + c2) / (t1 + t2)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / t1 + 1 / t2))
    if se == 0:
        return 0.0, 1.0
    z = (p1 - p2) / se
    # two-tailed p from standard normal (no scipy dependency)
    p = math.erfc(abs(z) / math.sqrt(2))
    return z, p


def stability_check_10K_1M(results):
    """results: output of run_sweep(sizes=("10K","1M"), seeds=(...))."""
    print("=== L_E adj_agree_pre: accuracy stability, 10K vs 1M ===")
    for seed in results["10K"]:
        c1, t1 = results["10K"][seed]['accuracy'].get(('E', 'adj_agree_pre'), (0, 0))
        c2, t2 = results["1M"][seed]['accuracy'].get(('E', 'adj_agree_pre'), (0, 0))
        z, p = two_proportion_z(c1, t1, c2, t2)
        acc1, acc2 = c1 / max(t1, 1), c2 / max(t2, 1)
        print(f"  seed {seed}: 10K={acc1:.3f} (n={t1})  1M={acc2:.3f} (n={t2})  "
              f"z={z:+.2f}  p={p:.3f}  "
              f"{'[flat — consistent with A2]' if p > 0.05 else '[MOVED — check leakage/interference]'}")

    print("\n=== D-E sharing score (adj_agree_pre): 10K vs 1M ===")
    for seed in results["10K"]:
        s1 = results["10K"][seed]['sharing'].get('adj_agree_pre', {}).get('DE')
        s2 = results["1M"][seed]['sharing'].get('adj_agree_pre', {}).get('DE')
        if s1 is None or s2 is None:
            print(f"  seed {seed}: DE score missing at one or both scales")
            continue
        print(f"  seed {seed}: 10K={s1:+.3f}  1M={s2:+.3f}  delta={s2-s1:+.3f}  "
              f"{'[DROPPED — candidate N* signature]' if s2 < s1 - 0.05 else '[flat/rose — no dissociation yet]'}")

    print("\nNote: this is a per-seed eyeball comparison, not a formal test —")
    print("check consistency across both seeds before reading anything into")
    print("a single delta (Lesson 5). For a proper noise floor to compare the")
    print("D-E delta against, use pearson_same_noise_floor(lang='D') from Cell 12.")


# Uncomment to run the two-point version instead of the full 7-size sweep:
sweep_results_2pt = run_sweep(seeds=(0, 1), num_workers=NUM_WORKERS, sizes=("10K", "1M"))
stability_check_10K_1M(sweep_results_2pt)


  Size: 10K

  -- seed 0 --
GPT: 79,008 params  (L=2 H=2 D=48)
  [resumed from checkpoint, skipped training]
=== Per-sub-computation accuracy (with Wilson 95% CI) ===

  adj_agree:
    L_A: 0.002  [0.001, 0.005]  (n=1658)
    L_B: 0.001  [0.000, 0.004]  (n=1921)
    L_C: 0.002  [0.001, 0.006]  (n=1497)

  adj_agree_pre:
    L_D: 0.114  [0.099, 0.131]  (n=1578)
    L_E: 0.048  [0.038, 0.060]  (n=1470)

  det:
    L_A: 0.407  [0.392, 0.422]  (n=4242)
    L_B: 0.198  [0.187, 0.210]  (n=4695)
    L_C: 0.992  [0.988, 0.994]  (n=3827)
    L_D: 0.400  [0.385, 0.416]  (n=3988)
    L_E: 0.990  [0.986, 0.993]  (n=3579)

  n_after_det:
    L_A: 0.080  [0.072, 0.089]  (n=4242)
    L_B: 0.084  [0.077, 0.092]  (n=4695)
    L_C: 0.038  [0.032, 0.044]  (n=3827)
    L_D: 0.030  [0.025, 0.035]  (n=3988)
    L_E: 0.040  [0.034, 0.047]  (n=3579)

  num:
    L_A: 0.494  [0.479, 0.509]  (n=4242)
    L_C: 0.506  [0.490, 0.521]  (n=3827)
    L_D: 0.508  [0.493, 0.524]  (n=3988)
    L_E: 0.511  [0.495, 0.528]

In [34]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13.6 — Stability check, redone at feature level (from checkpoints)
# ══════════════════════════════════════════════════════════════════════

def stability_check_feature_level(sizes=("10K", "1M"), seeds=(0, 1)):
    feat_acc = {}
    for size in sizes:
        for seed in seeds:
            loaded = load_checkpoint(size, seed)
            if loaded is None:
                print(f"  [missing checkpoint: {size}/seed{seed} — skipping]")
                continue
            model, _log = loaded
            print(f"\n-- {size} / seed {seed} --")
            feat_acc[(size, seed)] = eval_feature_accuracy(model, n=1000, seed=777 + seed)
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    print("\n=== L_E adj_agree_pre: FEATURE-LEVEL accuracy stability, 10K vs 1M ===")
    for seed in seeds:
        c1, t1 = feat_acc[(sizes[0], seed)].get(('E', 'adj_agree_pre'), (0, 1))
        c2, t2 = feat_acc[(sizes[1], seed)].get(('E', 'adj_agree_pre'), (0, 1))
        z, p = two_proportion_z(c1, t1, c2, t2)
        a1, a2 = c1 / max(t1, 1), c2 / max(t2, 1)
        print(f"  seed {seed}: {sizes[0]}={a1:.3f} (n={t1})  {sizes[1]}={a2:.3f} (n={t2})  "
              f"z={z:+.2f}  p={p:.3f}  "
              f"{'[flat — consistent with A2]' if p > 0.05 else '[MOVED — investigate]'}")
    return feat_acc

feat_stability = stability_check_feature_level()

GPT: 79,008 params  (L=2 H=2 D=48)

-- 10K / seed 0 --
=== Restricted-vocab feature accuracy (chance = 0.500), with Wilson CI ===

  adj_agree:
    L_A: 1.000  [0.998, 1.000]  (n=1976)
    L_B: 1.000  [0.998, 1.000]  (n=2492)
    L_C: 1.000  [0.998, 1.000]  (n=1871)

  adj_agree_pre:
    L_D: 1.000  [0.998, 1.000]  (n=2035)
    L_E: 0.494  [0.471, 0.518]  (n=1796)

  v_agree:
    L_A: 1.000  [0.998, 1.000]  (n=2287)
    L_B: 1.000  [0.999, 1.000]  (n=2773)
    L_C: 1.000  [0.998, 1.000]  (n=2144)
    L_D: 1.000  [0.998, 1.000]  (n=2499)
    L_E: 1.000  [0.998, 1.000]  (n=2170)

  v_agree_rel:
    L_A: 1.000  [0.992, 1.000]  (n=497)
    L_B: 1.000  [0.993, 1.000]  (n=586)
    L_C: 1.000  [0.992, 1.000]  (n=471)
    L_D: 1.000  [0.992, 1.000]  (n=482)
    L_E: 1.000  [0.990, 1.000]  (n=396)
GPT: 79,008 params  (L=2 H=2 D=48)

-- 10K / seed 1 --
=== Restricted-vocab feature accuracy (chance = 0.500), with Wilson CI ===

  adj_agree:
    L_A: 1.000  [0.998, 1.000]  (n=1877)
    L_B: 1.000 

In [16]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13.6 — Stability check, redone at feature level (from checkpoints)
# ══════════════════════════════════════════════════════════════════════

def stability_check_feature_level(sizes=("10K", "1M"), seeds=(0, 1)):
    feat_acc = {}
    for size in sizes:
        for seed in seeds:
            loaded = load_checkpoint(size, seed)
            if loaded is None:
                print(f"  [missing checkpoint: {size}/seed{seed} — skipping]")
                continue
            model, _log = loaded
            print(f"\n-- {size} / seed {seed} --")
            feat_acc[(size, seed)] = eval_feature_accuracy(model, n=1000, seed=777 + seed)
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    print("\n=== L_E adj_agree_pre: FEATURE-LEVEL accuracy stability, 10K vs 1M ===")
    for seed in seeds:
        c1, t1 = feat_acc[(sizes[0], seed)].get(('E', 'adj_agree_pre'), (0, 1))
        c2, t2 = feat_acc[(sizes[1], seed)].get(('E', 'adj_agree_pre'), (0, 1))
        z, p = two_proportion_z(c1, t1, c2, t2)
        a1, a2 = c1 / max(t1, 1), c2 / max(t2, 1)
        print(f"  seed {seed}: {sizes[0]}={a1:.3f} (n={t1})  {sizes[1]}={a2:.3f} (n={t2})  "
              f"z={z:+.2f}  p={p:.3f}  "
              f"{'[flat — consistent with A2]' if p > 0.05 else '[MOVED — investigate]'}")
    return feat_acc

feat_stability = stability_check_feature_level()

GPT: 79,008 params  (L=2 H=2 D=48)

-- 10K / seed 0 --
=== Restricted-vocab feature accuracy (chance = 0.500), with Wilson CI ===

  adj_agree:
    L_A: 1.000  [0.998, 1.000]  (n=1976)
    L_B: 1.000  [0.998, 1.000]  (n=2492)
    L_C: 1.000  [0.998, 1.000]  (n=1871)

  adj_agree_pre:
    L_D: 1.000  [0.998, 1.000]  (n=2035)
    L_E: 0.494  [0.471, 0.518]  (n=1796)

  v_agree:
    L_A: 1.000  [0.998, 1.000]  (n=2287)
    L_B: 1.000  [0.999, 1.000]  (n=2773)
    L_C: 1.000  [0.998, 1.000]  (n=2144)
    L_D: 1.000  [0.998, 1.000]  (n=2499)
    L_E: 1.000  [0.998, 1.000]  (n=2170)

  v_agree_rel:
    L_A: 1.000  [0.992, 1.000]  (n=497)
    L_B: 1.000  [0.993, 1.000]  (n=586)
    L_C: 1.000  [0.992, 1.000]  (n=471)
    L_D: 1.000  [0.992, 1.000]  (n=482)
    L_E: 1.000  [0.990, 1.000]  (n=396)
GPT: 79,008 params  (L=2 H=2 D=48)

-- 10K / seed 1 --
=== Restricted-vocab feature accuracy (chance = 0.500), with Wilson CI ===

  adj_agree:
    L_A: 1.000  [0.998, 1.000]  (n=1877)
    L_B: 1.000 

NameError: name 'two_proportion_z' is not defined

In [ ]:
# CELL 13 — Full N sweep with per-sub-comp sharing scores (multi-seed)
# ══════════════════════════════════════════════════════════════════════

SWEEP_SIZES = ["10K", "25K", "50K", "100K", "250K", "500K", "1M"]
SWEEP_STEPS = {
    "10K":  50000, "25K":  40000, "50K":  35000, "100K": 40000,
    "250K": 50000, "500K": 60000, "1M":   70000,
}
SUBCOMPS_TO_MEASURE = ['adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_rel',
                       'n_after_det', 'structural']
# NOTE on adj_agree_pre: only L_D and L_E have positions with this
# label, so sharing_scores_by_subcomp will populate ONLY the 'DE' pair
# for it (every other pair involving adj_agree_pre is None by
# construction — that's correct, not a bug).
#
# n_after_det remains the primary ε signal for the A/B/C triad per the
# v7 calibration; adj_agree_pre is the primary ε signal for the D/E
# pair. adj_agree (post-nominal)/v_agree likely saturate near ceiling
# (positions that can copy from already-generated context) and may not
# show sweep-trajectory movement — don't treat a flat line there as ε
# failing without checking n_after_det / adj_agree_pre first.
#
# THE HEADLINE CHECK for this chat's experiment (per Lesson 5, at
# least 2 seeds before trusting any single number):
#   - L_E's OWN adj_agree_pre accuracy: should stay pinned near 0.5 at
#     BOTH 10K and 1M (ε doesn't shrink with capacity — A2).
#   - The D–E sharing score on adj_agree_pre: THIS is what should move
#     with capacity if the theory is right — dissociating (dropping)
#     as N crosses N* for this pair, while L_E's raw accuracy stays
#     flat near chance the whole time.
#   - If L_E's accuracy stays flat AND the D–E sharing score also stays
#     flat: N* for this pair sits outside [10K, 1M] — evidence for a
#     wider sweep, not evidence against the theory.

# ══════════════════════════════════════════════════════════════════════
# CELL 13 (patched) — run_sweep with checkpointing/resume
# ══════════════════════════════════════════════════════════════════════

def run_sweep(seeds=(0, 1), num_workers=0, resume=True):
    """
    Multi-seed sweep with checkpoint/resume. Returns {size: {seed: {...}}}.
    If resume=True and a checkpoint exists for (size, seed), loads the
    trained model instead of retraining — only re-runs the (cheap) eval
    on top of it.
    """
    results = {size: {} for size in SWEEP_SIZES}
    for size_label in SWEEP_SIZES:
        print(f"\n{'='*60}")
        print(f"  Size: {size_label}")
        print(f"{'='*60}")
        cfg = SIZE_CONFIGS[size_label]

        for seed in seeds:
            print(f"\n  -- seed {seed} --")

            loaded = load_checkpoint(size_label, seed) if resume else None
            if loaded is not None:
                model, log = loaded
                print(f"  [resumed from checkpoint, skipped training]")
            else:
                model, log = train_model(
                    cfg, max_steps=SWEEP_STEPS[size_label],
                    eval_every=2000, seed=seed, num_workers=num_workers
                )
                save_checkpoint(model, cfg, log, size_label, seed)

            acc = eval_subcomp_accuracy(model, n=800, seed=777 + seed)
            sharing = sharing_scores_by_subcomp(
                model, subcomps=SUBCOMPS_TO_MEASURE, n_examples=512, seed=999 + seed
            )

            results[size_label][seed] = {
                'accuracy': dict(acc),
                'sharing': sharing,
                'val_loss': log['final_val'],
            }

            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    return results


def aggregate_sweep_results(results):
    """
    Collapse run_sweep()'s {size: {seed: {...}}} into
    {size: {'accuracy': {(lang,sc): (mean_acc, std_acc, mean_n)},
            'sharing':  {sc: {pair: (mean, std)}}}}
    so no single-seed number gets read on its own (Lesson 5).
    """
    agg = {}
    for size_label, by_seed in results.items():
        seeds = list(by_seed.keys())

        acc_keys = set()
        for s in seeds:
            acc_keys.update(by_seed[s]['accuracy'].keys())
        acc_agg = {}
        for key in acc_keys:
            accs, ns = [], []
            for s in seeds:
                c, t = by_seed[s]['accuracy'].get(key, (0, 0))
                if t > 0:
                    accs.append(c / t)
                    ns.append(t)
            if accs:
                acc_agg[key] = (float(np.mean(accs)), float(np.std(accs)), float(np.mean(ns)))

        sc_keys = set()
        for s in seeds:
            sc_keys.update(by_seed[s]['sharing'].keys())
        sharing_agg = {}
        for sc in sc_keys:
            pair_keys = set()
            for s in seeds:
                pair_keys.update(by_seed[s]['sharing'].get(sc, {}).keys())
            pair_agg = {}
            for pair in pair_keys:
                vals = [by_seed[s]['sharing'][sc][pair] for s in seeds
                        if by_seed[s]['sharing'].get(sc, {}).get(pair) is not None]
                if vals:
                    pair_agg[pair] = (float(np.mean(vals)), float(np.std(vals)))
                else:
                    pair_agg[pair] = (None, None)
            sharing_agg[sc] = pair_agg

        agg[size_label] = {'accuracy': acc_agg, 'sharing': sharing_agg}
    return agg

# Uncomment to run (expensive — trains len(SWEEP_SIZES) * len(seeds)
# models). Only run once the leakage audit (3.5) and calibration gate
# (Cell 10) have both passed.
# sweep_results = run_sweep(seeds=(0, 1), num_workers=NUM_WORKERS)
# sweep_agg = aggregate_sweep_results(sweep_results)


# ══════════════════════════════════════════════════════════════════════


In [20]:
# CELL 14 — Plots
# ══════════════════════════════════════════════════════════════════════

def plot_sweep(sweep_agg, subcomps=None, pairs=None, figsize=(14, 12)):
    """
    Uses aggregated (mean±std across seeds) sharing scores.
    One panel per sub-computation, with error bars across seeds.
    Highlights the DE pair (the ε test) wherever it's populated.
    """
    import matplotlib.pyplot as plt

    if subcomps is None:
        subcomps = SUBCOMPS_TO_MEASURE
    if pairs is None:
        pairs = [f"{a}{b}" for a, b in itertools.combinations(LANGS, 2)]

    sizes = list(sweep_agg.keys())
    x = np.arange(len(sizes))

    fig, axes = plt.subplots(len(subcomps), 1, figsize=figsize, sharex=True)
    if len(subcomps) == 1:
        axes = [axes]

    for ax_idx, sc in enumerate(subcomps):
        ax = axes[ax_idx]
        for pair in pairs:
            means, stds = [], []
            any_data = False
            for sz in sizes:
                m, s = sweep_agg[sz]['sharing'].get(sc, {}).get(pair, (None, None))
                means.append(m if m is not None else np.nan)
                stds.append(s if s is not None else 0.0)
                if m is not None:
                    any_data = True
            if not any_data:
                continue
            style = '-o' if pair != 'DE' else '-s'
            lw = 1.5 if pair != 'DE' else 2.5
            ax.errorbar(x, means, yerr=stds, fmt=style, label=pair,
                       markersize=5, linewidth=lw)
        ax.set_ylabel('Pearson')
        ax.set_title(sc, fontsize=10)
        ax.legend(fontsize=7, ncol=3)
        ax.set_ylim(-0.1, 1.05)
        ax.grid(alpha=0.3)

    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(sizes, rotation=45)
    axes[-1].set_xlabel('Model size')

    plt.tight_layout()
    plt.savefig('v8_sharing_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v8_sharing_sweep.png")


# ══════════════════════════════════════════════════════════════════════


In [21]:
# CELL 15 — Accuracy vs size plot (behavioral check)
# ══════════════════════════════════════════════════════════════════════

def plot_accuracy_sweep(sweep_agg, subcomps=None, figsize=(12, 10)):
    """
    Per-sub-comp accuracy (mean±std across seeds) for each language.
    Prediction for adj_agree_pre: L_D near ceiling and flat; L_E
    pinned near 0.5 and flat across scale (own accuracy shouldn't
    move — the D-E SHARING score is what should move, see plot_sweep).
    """
    import matplotlib.pyplot as plt

    if subcomps is None:
        subcomps = ['adj_agree_pre', 'n_after_det', 'v_agree', 'structural']

    sizes = list(sweep_agg.keys())
    x = np.arange(len(sizes))

    fig, axes = plt.subplots(len(subcomps), 1, figsize=figsize, sharex=True)
    if len(subcomps) == 1:
        axes = [axes]

    for ax_idx, sc in enumerate(subcomps):
        ax = axes[ax_idx]
        for L in LANGS:
            means, stds = [], []
            any_data = False
            for sz in sizes:
                entry = sweep_agg[sz]['accuracy'].get((L, sc))
                if entry is None:
                    means.append(np.nan)
                    stds.append(0.0)
                else:
                    m, s, _n = entry
                    means.append(m)
                    stds.append(s)
                    any_data = True
            if not any_data:
                continue
            ax.errorbar(x, means, yerr=stds, fmt='o-', label=f'L_{L}', markersize=4)
        if sc == 'adj_agree_pre':
            ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='chance')
        ax.set_ylabel('Accuracy')
        ax.set_title(sc, fontsize=10)
        ax.legend(fontsize=8)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=0.3)

    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(sizes, rotation=45)
    axes[-1].set_xlabel('Model size')

    plt.tight_layout()
    plt.savefig('v8_accuracy_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v8_accuracy_sweep.png")


# ══════════════════════════════════════════════════════════════════════


In [22]:
# CELL 16 — Diagnosis guide
# ══════════════════════════════════════════════════════════════════════

DIAGNOSIS = """
=== v8 Diagnosis Guide (L_E addition) ===

LEAKAGE AUDIT (Cell 3.5) — run first, before training anything:

IF max NUM->gender deviation > 0.03, or the cross-NP z-score has
|z| >= 3:
  → Possible leakage confound. Do NOT proceed to training until this
    is understood. Check:
    (a) Is gender really re-sampled with rng.choice(['M','F']) fresh
        at every sample_np call, with no dependency on outer state?
    (b) Increase audit n (leakage_audit_L_E(n=100000)) to rule out
        sampling noise before concluding there's a real leak.
    (c) If real: search for any place NUM_E or DET_E's value is
        conditioned on gender (it shouldn't be — both should be
        assigned independently of gender in sample_np).

CALIBRATION GATE (Cell 10):

IF L_D adj_agree_pre is NOT near ceiling (~1.0) at 10K:
  → DET_D's gender signal isn't being read at the ADJ position even
    though it's structurally available. Check whether P_ADJ is high
    enough for adj_agree_pre positions to appear often enough to
    learn from, or whether capacity is so low the model can't afford
    the DET_D→ADJ pathway at all yet (try increasing 10K's n_embd
    slightly as a diagnostic, not as the final config).

IF L_E adj_agree_pre CI excludes 0.5 already at 10K:
  → Either (a) a leakage confound the audit missed — increase audit n
    and recheck the specific mechanism above; or (b) genuinely check
    whether the model is instead exploiting SOME other channel not
    covered by the audit (e.g. embedding-space collisions between
    L_E's invariant DET token and gender-correlated positional
    statistics elsewhere in the packed sequence — try auditing at
    the packed-sequence level with n_examples >= 50K if (a) comes
    back clean).

SWEEP (Cell 13) — THE HEADLINE CHECK:

IF L_E adj_agree_pre accuracy stays flat ≈0.5 AND D-E sharing score
DROPS with scale:
  → This is the predicted N* signature. Report both curves together
    — the flat accuracy is what rules out "L_E secretly learned
    something"; the dropping sharing score is the actual finding.

IF L_E adj_agree_pre accuracy stays flat ≈0.5 AND D-E sharing score
ALSO stays flat:
  → N* for this (D,E) pair sits outside [10K, 1M]. This is NOT
    evidence against the theory — extend the sweep (e.g. add a 2M or
    4M point) before concluding anything either way.

IF L_E adj_agree_pre accuracy RISES with scale:
  → Should not happen if the leakage audit and calibration gate both
    passed clean — if it does anyway, treat it as a "found an
    unintended shortcut" result, not an ε-resisting-capacity result.
    Re-run the leakage audit at the SPECIFIC scale where accuracy
    rose (bigger models can exploit subtler regularities than the
    audit's simple statistics catch) before reporting this as
    anything.

ALWAYS, before trusting either headline number:
  (a) check both seeds individually, not just the mean (Lesson 5);
  (b) check n per branch — sample sizes can be lopsided by construction
      since adj_agree_pre only fires on the P_ADJ branch (Lesson 4);
  (c) treat Pearson activation-profile as secondary to the direct
      accuracy numbers (Lesson 3) — accuracy is the primary evidence.

HEADLINE PLOT target (P1 for the L_E arm of the paper):
  → L_E adj_agree_pre accuracy: FLAT near 0.5 across all N.
  → L_D adj_agree_pre accuracy: FLAT near ceiling across all N.
  → D-E sharing score on adj_agree_pre: DECREASES as N grows (the N*
    crossover signature), OR stays flat with an explicit note that
    N* lies outside the swept range.
"""
print(DIAGNOSIS)



=== v8 Diagnosis Guide (L_E addition) ===

LEAKAGE AUDIT (Cell 3.5) — run first, before training anything:

IF max NUM->gender deviation > 0.03, or the cross-NP z-score has
|z| >= 3:
  → Possible leakage confound. Do NOT proceed to training until this
    is understood. Check:
    (a) Is gender really re-sampled with rng.choice(['M','F']) fresh
        at every sample_np call, with no dependency on outer state?
    (b) Increase audit n (leakage_audit_L_E(n=100000)) to rule out
        sampling noise before concluding there's a real leak.
    (c) If real: search for any place NUM_E or DET_E's value is
        conditioned on gender (it shouldn't be — both should be
        assigned independently of gender in sample_np).

CALIBRATION GATE (Cell 10):

IF L_D adj_agree_pre is NOT near ceiling (~1.0) at 10K:
  → DET_D's gender signal isn't being read at the ADJ position even
    though it's structurally available. Check whether P_ADJ is high
    enough for adj_agree_pre positions to appear 

In [27]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6a-pre — pearson (self-contained; identical to Cell 11's,
# redefined here so this file doesn't silently depend on Cell 11
# having been run — Cell 11's own instrument is a documented dead end)
# ══════════════════════════════════════════════════════════════════════

def pearson(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a**2).sum() * (b**2).sum())
    if denom < 1e-12:
        return 0.0
    return float((a * b).sum() / denom)

In [28]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6a — Reference means (mean-ablation replacement values)
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def compute_reference_means(model, lang, subcomp, n_examples=512, seed=1234):
    """
    Per-(language, subcomp) mean activation for every attention head's
    pre-c_proj slice and every MLP layer's post-c_proj output (the
    vector actually added to the residual stream — NOT blk.mlp's
    internal hidden layer, which is what _last_neuron stores elsewhere
    in this notebook; that's the wrong tensor to ablate here since
    it's pre-projection).

    Returns {'head_mean': (n_layer, n_head, head_dim),
             'mlp_mean':  (n_layer, n_embd)}  (both on CPU)
    or None if lang has no subcomp positions.
    """
    model.eval()
    cfg = model.cfg
    head_dim = cfg.n_embd // cfg.n_head
    rng = random.Random(seed)

    head_sum = torch.zeros(cfg.n_layer, cfg.n_head, head_dim, device=device)
    mlp_sum = torch.zeros(cfg.n_layer, cfg.n_embd, device=device)
    n_pos = 0

    # blk.mlp._last_neuron (used elsewhere in this notebook) is the
    # PRE-c_proj hidden layer (B,T,4*n_embd) — not what gets added to
    # the residual. Capture the module's actual return value instead.
    mlp_outputs = [None] * cfg.n_layer

    def make_mlp_capture(l):
        def hook(module, inp, out):
            mlp_outputs[l] = out.detach()
        return hook

    handles = [blk.mlp.register_forward_hook(make_mlp_capture(l))
               for l, blk in enumerate(model.blocks)]

    all_ids, all_positions = [], []
    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp and 1 <= i < MAX_LEN - 1]
        if positions:
            all_ids.append(ids[:-1])
            all_positions.append(positions)

    if not all_ids:
        for h in handles:
            h.remove()
        return None

    batch_size = 64
    for start in range(0, len(all_ids), batch_size):
        batch_ids = all_ids[start:start + batch_size]
        batch_pos = all_positions[start:start + batch_size]
        x = torch.tensor(batch_ids, dtype=torch.long, device=device)
        _ = model(x)  # populates blk.attn._last_head_out + mlp_outputs

        for b_idx, positions in enumerate(batch_pos):
            for p in positions:
                for l, blk in enumerate(model.blocks):
                    head_out = blk.attn._last_head_out  # (B,H,T,Dh)
                    head_sum[l] += head_out[b_idx, :, p, :]
                    mlp_sum[l] += mlp_outputs[l][b_idx, p, :]
                n_pos += 1

    for h in handles:
        h.remove()

    if n_pos == 0:
        return None

    return {
        'head_mean': (head_sum / n_pos).cpu(),
        'mlp_mean': (mlp_sum / n_pos).cpu(),
    }

In [29]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6b — Ablation hooks
# ══════════════════════════════════════════════════════════════════════

def _ablate_head_pre_hook(head_idx, head_dim, replacement):
    """
    forward_pre_hook on blk.attn.c_proj. c_proj's input is the
    concatenated per-head output (B,T,C) — head h occupies the
    contiguous slice [h*head_dim:(h+1)*head_dim] because the forward
    pass builds it via out.transpose(1,2).contiguous().view(B,T,C)
    from (B,H,T,Dh). Replacing that slice with its lang-specific mean
    (broadcast over batch and time) ablates exactly head h's
    contribution before it's mixed into the residual by c_proj.
    """
    def pre_hook(module, inputs):
        x = inputs[0].clone()
        s = head_idx * head_dim
        e = s + head_dim
        x[:, :, s:e] = replacement.to(x.dtype).to(x.device)
        return (x,)
    return pre_hook


def _ablate_mlp_hook(replacement):
    """
    forward_hook on blk.mlp (the whole module). Replaces the MLP's
    entire output — the vector actually added to the residual stream —
    with its lang-specific mean, broadcast over batch and time. This
    ablates the whole MLP layer as one component, per your granularity
    choice (not per-neuron within it).
    """
    def hook(module, inp, out):
        rep = replacement.to(out.dtype).to(out.device)
        return rep.expand_as(out).contiguous()
    return hook

In [30]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6c — Importance vectors
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def ckic_importance_vector(model, lang, subcomp, n_examples=512,
                           eval_seed=5678, mean_n=512, mean_seed=1234):
    """
    For each attention head and each MLP layer, mean-ablate it (using
    lang's own subcomp-matched mean as the replacement) and measure
    mean KL(original || ablated) on the output distribution at the
    subcomp-matched positions. This is the component's importance to
    lang's computation of subcomp.

    Component order is FIXED and identical across languages (same
    architecture): layer 0 head 0..H-1, layer 0 mlp, layer 1 head
    0..H-1, layer 1 mlp, ... — this is what makes index-wise
    correlation between imp_D and imp_E meaningful.

    Returns (importance_vector: np.ndarray, labels: list, n_positions: int)
    or (None, None, 0) if lang has no subcomp positions. n_positions is
    the number of subcomp-matched (sequence, position) pairs the KL
    estimate is averaged over — report it; adj_agree_pre only fires on
    the has_adj branch (P_ADJ=0.40 per NP), so this can be lopsided
    across languages/scales (Lesson 4).
    """
    cfg = model.cfg
    head_dim = cfg.n_embd // cfg.n_head

    ref = compute_reference_means(model, lang, subcomp,
                                  n_examples=mean_n, seed=mean_seed)
    if ref is None:
        return None, None
    head_mean = ref['head_mean'].to(device)   # (n_layer, n_head, head_dim)
    mlp_mean = ref['mlp_mean'].to(device)     # (n_layer, n_embd)

    rng = random.Random(eval_seed)
    all_ids, all_positions = [], []
    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp and 1 <= i < MAX_LEN - 1]
        if positions:
            all_ids.append(ids[:-1])
            all_positions.append(positions)

    if not all_ids:
        return None, None, 0

    def gather_logp():
        """Forward all eval examples through the (possibly hooked)
        model; return log-probs at the position PREDICTING each
        matched token (index p-1 predicts token at index p), stacked
        as (n_positions, V)."""
        out = []
        batch_size = 64
        for start in range(0, len(all_ids), batch_size):
            batch_ids = all_ids[start:start + batch_size]
            batch_pos = all_positions[start:start + batch_size]
            x = torch.tensor(batch_ids, dtype=torch.long, device=device)
            logits = model(x)
            logp = F.log_softmax(logits, dim=-1)
            for b_idx, positions in enumerate(batch_pos):
                for p in positions:
                    out.append(logp[b_idx, p - 1])
        return torch.stack(out)

    baseline_logp = gather_logp()
    n_positions = baseline_logp.shape[0]

    labels, importances = [], []

    for l, blk in enumerate(model.blocks):
        for h in range(cfg.n_head):
            handle = blk.attn.c_proj.register_forward_pre_hook(
                _ablate_head_pre_hook(h, head_dim, head_mean[l, h]))
            ablated_logp = gather_logp()
            handle.remove()
            kl = (baseline_logp.exp() *
                  (baseline_logp - ablated_logp)).sum(-1).mean().item()
            importances.append(kl)
            labels.append((l, 'head', h))

        handle = blk.mlp.register_forward_hook(_ablate_mlp_hook(mlp_mean[l]))
        ablated_logp = gather_logp()
        handle.remove()
        kl = (baseline_logp.exp() *
              (baseline_logp - ablated_logp)).sum(-1).mean().item()
        importances.append(kl)
        labels.append((l, 'mlp', None))

    return np.array(importances), labels, n_positions


def _rankdata(a):
    """Ranks with ties averaged (simple, adequate for continuous KL
    values where exact ties are rare)."""
    a = np.asarray(a)
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty(len(a), dtype=float)
    ranks[order] = np.arange(len(a))
    # average tied ranks
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sorted_a[j + 1] == sorted_a[i]:
            j += 1
        if j > i:
            avg_rank = ranks[order[i:j + 1]].mean()
            ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def spearman(a, b):
    """Rank correlation — far less swayed by a single outlier magnitude
    than Pearson, which matters here: importance vectors are often
    dominated by one or two large components (see mean-KL rankings)."""
    return pearson(_rankdata(a), _rankdata(b))


def print_component_ranking(imp, labels, top_k=5, name=""):
    order = np.argsort(imp)[::-1]
    print(f"    top-{top_k} components by importance{f' ({name})' if name else ''}:")
    for idx in order[:top_k]:
        l, kind, h = labels[idx]
        tag = f"L{l}/head{h}" if kind == 'head' else f"L{l}/mlp"
        print(f"      {tag:<12} KL={imp[idx]:.5f}")

In [31]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6d — D-E CKIC, with generic-convergence control
# ══════════════════════════════════════════════════════════════════════

def ckic_de(model, target_subcomp='adj_agree_pre', control_subcomp='v_agree',
            n_examples=512, eval_seed=5678, mean_n=512, mean_seed=1234,
            verbose=True):
    """
    CKIC(D,E) on target_subcomp and control_subcomp for one model.
    v_agree is the ε≈0 control (both D and E succeed at it; sharing
    is expected regardless of scale) — a target CKIC drop that
    exceeds the control's drop is the ε-specific dissociation signal,
    same logic as the generic-convergence control used for the
    (failed) linear probe in Cell 11.5.
    """
    results = {}
    for sc in (target_subcomp, control_subcomp):
        imp_D, labels_D, n_D = ckic_importance_vector(
            model, 'D', sc, n_examples=n_examples, eval_seed=eval_seed,
            mean_n=mean_n, mean_seed=mean_seed)
        imp_E, labels_E, n_E = ckic_importance_vector(
            model, 'E', sc, n_examples=n_examples, eval_seed=eval_seed,
            mean_n=mean_n, mean_seed=mean_seed)
        if imp_D is None or imp_E is None:
            print(f"  [{sc}] missing subcomp positions for D or E — skipping")
            results[sc] = None
            continue
        assert labels_D == labels_E, "component ordering mismatch between D and E"

        r_pearson = pearson(imp_D, imp_E)
        r_spearman = spearman(imp_D, imp_E)
        results[sc] = {
            'imp_D': imp_D, 'imp_E': imp_E, 'labels': labels_D,
            'ckic': r_pearson, 'ckic_spearman': r_spearman,
            'mean_imp_D': float(np.mean(imp_D)),
            'mean_imp_E': float(np.mean(imp_E)),
            'n_D': n_D, 'n_E': n_E,
        }
        if verbose:
            print(f"  [{sc}] n_positions: D={n_D}  E={n_E}   "
                  f"({len(imp_D)} components)")
            print(f"  [{sc}] CKIC(D,E) Pearson={r_pearson:+.3f}  "
                  f"Spearman={r_spearman:+.3f}   "
                  f"(mean|imp_D|={np.mean(imp_D):.5f}, "
                  f"mean|imp_E|={np.mean(imp_E):.5f})")
            if abs(r_pearson - r_spearman) > 0.15:
                print(f"    NOTE: Pearson/Spearman disagree by "
                      f"{abs(r_pearson - r_spearman):.3f} — the Pearson "
                      f"number is likely being driven by one or two "
                      f"outlier-magnitude components rather than overall "
                      f"rank agreement. Trust Spearman more here.")
            if np.mean(imp_E) < 1e-5:
                print(f"    WARNING: E's mean importance on {sc} is ~0 — "
                      f"E may have NO allocated computation at this "
                      f"position (CKIC would be meaningless noise, not "
                      f"a low-sharing signal). Check mean|imp_E| on "
                      f"{control_subcomp} to confirm E has learnable "
                      f"circuitry the ablation methodology can detect "
                      f"at all before trusting this number.")
            print_component_ranking(imp_D, labels_D, name=f"D, {sc}")
            print_component_ranking(imp_E, labels_D, name=f"E, {sc}")
    return results

In [32]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6e — Scale sweep
# ══════════════════════════════════════════════════════════════════════

def ckic_sweep(sizes=("10K", "1M"), seeds=(0, 1),
               target_subcomp='adj_agree_pre', control_subcomp='v_agree',
               n_examples=512, eval_seed=5678, mean_n=512, mean_seed=1234):
    """
    Runs ckic_de at each (size, seed), then reports whether the
    target's CKIC drop across scale EXCEEDS the control's drop —
    mirrors the reporting structure of the (superseded)
    dissociation_with_control in Cell 11.5.
    """
    results = defaultdict(dict)

    for size in sizes:
        for seed in seeds:
            loaded = load_checkpoint(size, seed)
            if loaded is None:
                print(f"  [missing checkpoint: {size}/seed{seed} — skipping]")
                continue
            model, _log = loaded
            print(f"\n{size} / seed {seed}:")
            r = ckic_de(model, target_subcomp=target_subcomp,
                       control_subcomp=control_subcomp,
                       n_examples=n_examples, eval_seed=eval_seed,
                       mean_n=mean_n, mean_seed=mean_seed)
            results[(size, seed)] = r
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    print(f"\n=== CKIC scale trend: {target_subcomp} vs {control_subcomp} ===")
    for seed in seeds:
        r10 = results.get((sizes[0], seed))
        r1m = results.get((sizes[1], seed))
        if not r10 or not r1m:
            print(f"  seed {seed}: missing data, skipping")
            continue
        t10, t1m = r10.get(target_subcomp), r1m.get(target_subcomp)
        c10, c1m = r10.get(control_subcomp), r1m.get(control_subcomp)
        if not all([t10, t1m, c10, c1m]):
            print(f"  seed {seed}: missing target/control data, skipping")
            continue

        print(f"  seed {seed}:")
        for stat in ('ckic', 'ckic_spearman'):
            label = "Pearson " if stat == 'ckic' else "Spearman"
            t_delta = t1m[stat] - t10[stat]
            c_delta = c1m[stat] - c10[stat]
            specific = t_delta - c_delta
            print(f"    [{label}] {target_subcomp}: {t10[stat]:+.3f} -> "
                  f"{t1m[stat]:+.3f}  (Δ={t_delta:+.3f})   "
                  f"{control_subcomp}: {c10[stat]:+.3f} -> {c1m[stat]:+.3f}  "
                  f"(Δ={c_delta:+.3f})   ε-specific: {specific:+.3f}")
        # Verdict keyed off Spearman (less outlier-sensitive) — see NOTE
        # above if it disagrees materially with Pearson.
        t_delta_s = t1m['ckic_spearman'] - t10['ckic_spearman']
        c_delta_s = c1m['ckic_spearman'] - c10['ckic_spearman']
        specific_s = t_delta_s - c_delta_s
        if specific_s < -0.05:
            verdict = "[DISSOCIATION beyond generic drift — the N* signature]"
        elif abs(specific_s) <= 0.05:
            verdict = "[flat relative to control — N* may sit outside swept range]"
        else:
            verdict = "[unexpected: target shares MORE than control with scale]"
        print(f"    verdict (Spearman-based): {verdict}")

    return results

In [33]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11.6f — Diagnosis guide
# ══════════════════════════════════════════════════════════════════════

CKIC_DIAGNOSIS = """
=== CKIC Diagnosis Guide ===

BEFORE TRUSTING ANY CKIC NUMBER:

IF mean|imp_E| on adj_agree_pre is ~0 (see WARNING in ckic_de output):
  -> E has no ablatable computation at this position at all. CKIC is
     undefined, not "low sharing". Check mean|imp_E| on v_agree for
     the SAME model — if that's also ~0, something is wrong with the
     ablation methodology itself (or the model), not with sharing.
     If v_agree's mean|imp_E| is healthy but adj_agree_pre's isn't,
     that's itself informative (E has been allocated zero resources
     for this position) but needs to be reported as that finding, not
     folded into a CKIC number.

IF the per-component rankings (print_component_ranking) for D and E
share almost no top components even at low N:
  -> Consistent with genuine dissociation already at the smallest
     scale swept — N* may be below 10K for this pair. Consider adding
     a smaller size point.

IF CKIC(D,E) on adj_agree_pre is high AND CKIC on v_agree is ALSO
high, at every scale:
  -> Both flat. Check the specific-component delta (Δtarget - Δcontrol)
     before concluding no dissociation — a small but real ε-specific
     effect can be masked if control CKIC is also drifting for
     unrelated reasons (e.g. general capacity-driven reorganization
     unrelated to this pair).

HEADLINE PLOT target (same shape as the accuracy sweep, now for the
mechanistic half):
  -> adj_agree_pre CKIC(D,E): DECREASES with scale (or flat, with an
     explicit note that N* lies outside [10K, 1M]).
  -> v_agree CKIC(D,E): stays roughly flat/high across scale — the
     control confirms the drop above is subcomp-specific, not a
     generic representational drift with model capacity.

IF Pearson and Spearman CKIC disagree by more than ~0.15 (flagged
automatically in ckic_de's output):
  -> Pearson is being driven by one or two outlier-magnitude
     components (common here — one dominant MLP or head often
     dwarfs everything else). Trust the Spearman-based verdict in
     ckic_sweep's output over the raw Pearson number; investigate
     the specific outlier component via print_component_ranking
     before writing up either number.

AT 10K (only 6 components: n_layer=2 * (n_head=2 + 1)):
  -> Both target and control CKIC can look near-ceiling for a boring
     reason — MLPs may simply dominate over heads for ANY local
     token prediction in a 2-layer model, regardless of ε-specific
     sharing. Absolute 10K values are not strong evidence either way
     on their own; the DELTA relative to the control across scale is
     what the argument rests on. Do not report a bare "CKIC=0.85 at
     10K" as evidence of sharing without this caveat attached.

ALWAYS, per Lesson 5: check both seeds individually before reporting
a mean; sample-size caveats (Lesson 4) apply to n_examples here the
same way they did to accuracy — adj_agree_pre only fires on the
P_ADJ branch, so raise n_examples if the printed n_positions (D and
E, both now reported by ckic_de) looks thin, especially for L_E
whose adj_agree_pre positions require the same has_adj branch as
L_D's.
"""
print(CKIC_DIAGNOSIS)


=== CKIC Diagnosis Guide ===

BEFORE TRUSTING ANY CKIC NUMBER:

IF mean|imp_E| on adj_agree_pre is ~0 (see WARNING in ckic_de output):
  -> E has no ablatable computation at this position at all. CKIC is
     undefined, not "low sharing". Check mean|imp_E| on v_agree for
     the SAME model — if that's also ~0, something is wrong with the
     ablation methodology itself (or the model), not with sharing.
     If v_agree's mean|imp_E| is healthy but adj_agree_pre's isn't,
     that's itself informative (E has been allocated zero resources
     for this position) but needs to be reported as that finding, not
     folded into a CKIC number.

IF the per-component rankings (print_component_ranking) for D and E
share almost no top components even at low N:
  -> Consistent with genuine dissociation already at the smallest
     scale swept — N* may be below 10K for this pair. Consider adding
     a smaller size point.

IF CKIC(D,E) on adj_agree_pre is high AND CKIC on v_agree is ALSO
high, at

In [34]:
# Example usage once checkpoints exist:
ckic_results = ckic_sweep(sizes=("10K", "1M"), seeds=(0, 1))

GPT: 79,008 params  (L=2 H=2 D=48)

10K / seed 0:
  [adj_agree_pre] n_positions: D=4939  E=4939   (6 components)
  [adj_agree_pre] CKIC(D,E) Pearson=+0.850  Spearman=+0.943   (mean|imp_D|=2.02506, mean|imp_E|=0.52343)
    top-5 components by importance (D, adj_agree_pre):
      L0/mlp       KL=7.21406
      L1/mlp       KL=3.64519
      L1/head1     KL=1.26421
      L0/head1     KL=0.02343
      L0/head0     KL=0.00349
    top-5 components by importance (E, adj_agree_pre):
      L1/mlp       KL=1.73322
      L0/mlp       KL=1.40576
      L1/head1     KL=0.00107
      L0/head1     KL=0.00036
      L0/head0     KL=0.00013
  [v_agree] n_positions: D=6009  E=6009   (6 components)
  [v_agree] CKIC(D,E) Pearson=+0.996  Spearman=+1.000   (mean|imp_D|=3.60210, mean|imp_E|=3.73481)
    top-5 components by importance (D, v_agree):
      L0/mlp       KL=7.80460
      L1/mlp       KL=6.58388
      L1/head1     KL=4.22611
      L0/head1     KL=2.98084
      L0/head0     KL=0.01679
    top-5 compone

In [37]:
#!/usr/bin/env python3
"""
Multilingual PCFG toy model — v8 addon: Component-Knockout Importance
Correlation (CKIC)
=======================================================================
Replaces the two failed sharing/dissociation instruments (Cell 11
Pearson activation-magnitude profiles, Cell 11.5 linear separability
probe) with a causal, ablation-based measure that isn't confounded by
either generic slot-magnitude convergence or trivial lexical
separability from disjoint vocabularies.

WHY THE PRIOR TWO FAILED (for reference — see learnings.md):
  - Pearson on |activation| profiles measures HOW HARD components
    fire, not WHAT they compute. D-E correlation rose with scale —
    opposite the predicted direction — diagnosed as tracking a
    generic ADJ-slot intensity reflex, not circuit sharing.
  - Linear probe on raw activations saturated at 1.000 for every
    subcomp (including an unrelated structural control) because D/E
    have fully disjoint vocabularies feeding the residual stream via
    embedding skip connections — it separates on lexical identity,
    not circuit structure.

CKIC ASKS A DIFFERENT QUESTION: for a given sub-computation (e.g.
adj_agree_pre), does knocking out component c change what the model
predicts, and — critically — is the SET of components that matter for
D the same as the set that matters for E? This is causal (ablation,
not correlation of raw magnitudes) and lexically invariant (KL on the
output *distribution*, not raw activations, so token-identity leakage
through skip connections doesn't inflate the signal).

METHOD:
  1. For language L and sub-computation S, compute each component's
     (attention head, or whole MLP layer) L-specific mean activation
     at S-matched positions — this is the "mean ablation" replacement
     value: it erases this-example-specific information while
     preserving the typical magnitude for this (language, position)
     pair.
  2. For each component c, mean-ablate c and measure the resulting
     mean KL(original || ablated) in the output distribution at the
     S-matched positions. This is c's importance to L's computation
     of S.
  3. Importance vectors imp_D and imp_E share a fixed component
     ordering (same architecture, same model — just different input
     language). CKIC(D,E) = Pearson(imp_D, imp_E).
  4. Run on adj_agree_pre (target, ε binds) and v_agree (control,
     ε≈0 — sharing expected at all scales). The target CKIC dropping
     relative to the control CKIC across scale is the N* signature;
     both flat means N* sits outside the swept range.

GRANULARITY (per your instruction): attention HEAD level (not
per-neuron within a head) + whole MLP LAYER level (not per-neuron
within the MLP's hidden layer). Component count per model:
n_layer * (n_head + 1) — small enough that ablating one at a time is
cheap for these toy-model sizes.

EDGE CASE WORTH WATCHING: if E's importance vector on adj_agree_pre is
near-zero everywhere (nothing ablatable actually changes E's output),
CKIC is undefined/meaningless — that would mean E has no allocated
computation at this position at all, not "different but real"
computation. Cell (c) below prints mean|imp_E| specifically so this is
visible before trusting any CKIC number. If it happens, check mean|imp_E|
on v_agree for the same model as a sanity check that E has SOME
learned circuitry the ablation methodology can detect at all.

Cell boundaries marked with # ══════ CELL N ══════ — paste after the
existing v8 Cell 11.5 (this does not require Cell 11.5's probe to be
run first, but assumes Cells 1-10 — grammar, model classes,
load_checkpoint — are already in the notebook's namespace). `pearson`
is redefined locally below (Cell 11.6a-pre) rather than assumed from
Cell 11, since Cell 11's Pearson-profile instrument is a documented
dead end you may not have run.
"""

# ══════════════════════════════════════════════════════════════════════
# CELL 11.6a-pre — pearson (self-contained; identical to Cell 11's,
# redefined here so this file doesn't silently depend on Cell 11
# having been run — Cell 11's own instrument is a documented dead end)
# ══════════════════════════════════════════════════════════════════════

def pearson(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a**2).sum() * (b**2).sum())
    if denom < 1e-12:
        return 0.0
    return float((a * b).sum() / denom)


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6a — Reference means (mean-ablation replacement values)
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def compute_reference_means(model, lang, subcomp, n_examples=512, seed=1234):
    """
    Per-(language, subcomp) mean activation for every attention head's
    pre-c_proj slice and every MLP layer's post-c_proj output (the
    vector actually added to the residual stream — NOT blk.mlp's
    internal hidden layer, which is what _last_neuron stores elsewhere
    in this notebook; that's the wrong tensor to ablate here since
    it's pre-projection).

    Returns {'head_mean': (n_layer, n_head, head_dim),
             'mlp_mean':  (n_layer, n_embd)}  (both on CPU)
    or None if lang has no subcomp positions.
    """
    model.eval()
    cfg = model.cfg
    head_dim = cfg.n_embd // cfg.n_head
    rng = random.Random(seed)

    head_sum = torch.zeros(cfg.n_layer, cfg.n_head, head_dim, device=device)
    mlp_sum = torch.zeros(cfg.n_layer, cfg.n_embd, device=device)
    n_pos = 0

    # blk.mlp._last_neuron (used elsewhere in this notebook) is the
    # PRE-c_proj hidden layer (B,T,4*n_embd) — not what gets added to
    # the residual. Capture the module's actual return value instead.
    mlp_outputs = [None] * cfg.n_layer

    def make_mlp_capture(l):
        def hook(module, inp, out):
            mlp_outputs[l] = out.detach()
        return hook

    handles = [blk.mlp.register_forward_hook(make_mlp_capture(l))
               for l, blk in enumerate(model.blocks)]

    all_ids, all_positions = [], []
    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp and 1 <= i < MAX_LEN - 1]
        if positions:
            all_ids.append(ids[:-1])
            all_positions.append(positions)

    if not all_ids:
        for h in handles:
            h.remove()
        return None

    batch_size = 64
    for start in range(0, len(all_ids), batch_size):
        batch_ids = all_ids[start:start + batch_size]
        batch_pos = all_positions[start:start + batch_size]
        x = torch.tensor(batch_ids, dtype=torch.long, device=device)
        _ = model(x)  # populates blk.attn._last_head_out + mlp_outputs

        for b_idx, positions in enumerate(batch_pos):
            for p in positions:
                for l, blk in enumerate(model.blocks):
                    head_out = blk.attn._last_head_out  # (B,H,T,Dh)
                    head_sum[l] += head_out[b_idx, :, p, :]
                    mlp_sum[l] += mlp_outputs[l][b_idx, p, :]
                n_pos += 1

    for h in handles:
        h.remove()

    if n_pos == 0:
        return None

    return {
        'head_mean': (head_sum / n_pos).cpu(),
        'mlp_mean': (mlp_sum / n_pos).cpu(),
    }


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6b — Ablation hooks
# ══════════════════════════════════════════════════════════════════════

def _ablate_head_pre_hook(head_idx, head_dim, replacement):
    """
    forward_pre_hook on blk.attn.c_proj. c_proj's input is the
    concatenated per-head output (B,T,C) — head h occupies the
    contiguous slice [h*head_dim:(h+1)*head_dim] because the forward
    pass builds it via out.transpose(1,2).contiguous().view(B,T,C)
    from (B,H,T,Dh). Replacing that slice with its lang-specific mean
    (broadcast over batch and time) ablates exactly head h's
    contribution before it's mixed into the residual by c_proj.
    """
    def pre_hook(module, inputs):
        x = inputs[0].clone()
        s = head_idx * head_dim
        e = s + head_dim
        x[:, :, s:e] = replacement.to(x.dtype).to(x.device)
        return (x,)
    return pre_hook


def _ablate_mlp_hook(replacement):
    """
    forward_hook on blk.mlp (the whole module). Replaces the MLP's
    entire output — the vector actually added to the residual stream —
    with its lang-specific mean, broadcast over batch and time. This
    ablates the whole MLP layer as one component, per your granularity
    choice (not per-neuron within it).
    """
    def hook(module, inp, out):
        rep = replacement.to(out.dtype).to(out.device)
        return rep.expand_as(out).contiguous()
    return hook


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6c — Importance vectors
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def ckic_importance_vector(model, lang, subcomp, n_examples=512,
                           eval_seed=5678, mean_n=512, mean_seed=1234):
    """
    For each attention head and each MLP layer, mean-ablate it (using
    lang's own subcomp-matched mean as the replacement) and measure
    mean KL(original || ablated) on the output distribution at the
    subcomp-matched positions. This is the component's importance to
    lang's computation of subcomp.

    Component order is FIXED and identical across languages (same
    architecture): layer 0 head 0..H-1, layer 0 mlp, layer 1 head
    0..H-1, layer 1 mlp, ... — this is what makes index-wise
    correlation between imp_D and imp_E meaningful.

    Returns (importance_vector: np.ndarray, labels: list, n_positions: int)
    or (None, None, 0) if lang has no subcomp positions. n_positions is
    the number of subcomp-matched (sequence, position) pairs the KL
    estimate is averaged over — report it; adj_agree_pre only fires on
    the has_adj branch (P_ADJ=0.40 per NP), so this can be lopsided
    across languages/scales (Lesson 4).
    """
    cfg = model.cfg
    head_dim = cfg.n_embd // cfg.n_head

    ref = compute_reference_means(model, lang, subcomp,
                                  n_examples=mean_n, seed=mean_seed)
    if ref is None:
        return None, None
    head_mean = ref['head_mean'].to(device)   # (n_layer, n_head, head_dim)
    mlp_mean = ref['mlp_mean'].to(device)     # (n_layer, n_embd)

    rng = random.Random(eval_seed)
    all_ids, all_positions = [], []
    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots)
                     if a['subcomp'] == subcomp and 1 <= i < MAX_LEN - 1]
        if positions:
            all_ids.append(ids[:-1])
            all_positions.append(positions)

    if not all_ids:
        return None, None, 0

    def gather_logp():
        """Forward all eval examples through the (possibly hooked)
        model; return log-probs at the position PREDICTING each
        matched token (index p-1 predicts token at index p), stacked
        as (n_positions, V)."""
        out = []
        batch_size = 64
        for start in range(0, len(all_ids), batch_size):
            batch_ids = all_ids[start:start + batch_size]
            batch_pos = all_positions[start:start + batch_size]
            x = torch.tensor(batch_ids, dtype=torch.long, device=device)
            logits = model(x)
            logp = F.log_softmax(logits, dim=-1)
            for b_idx, positions in enumerate(batch_pos):
                for p in positions:
                    out.append(logp[b_idx, p - 1])
        return torch.stack(out)

    baseline_logp = gather_logp()
    n_positions = baseline_logp.shape[0]

    labels, importances = [], []

    for l, blk in enumerate(model.blocks):
        for h in range(cfg.n_head):
            handle = blk.attn.c_proj.register_forward_pre_hook(
                _ablate_head_pre_hook(h, head_dim, head_mean[l, h]))
            ablated_logp = gather_logp()
            handle.remove()
            kl = (baseline_logp.exp() *
                  (baseline_logp - ablated_logp)).sum(-1).mean().item()
            importances.append(kl)
            labels.append((l, 'head', h))

        handle = blk.mlp.register_forward_hook(_ablate_mlp_hook(mlp_mean[l]))
        ablated_logp = gather_logp()
        handle.remove()
        kl = (baseline_logp.exp() *
              (baseline_logp - ablated_logp)).sum(-1).mean().item()
        importances.append(kl)
        labels.append((l, 'mlp', None))

    return np.array(importances), labels, n_positions


def _rankdata(a):
    """Ranks with ties averaged (simple, adequate for continuous KL
    values where exact ties are rare)."""
    a = np.asarray(a)
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty(len(a), dtype=float)
    ranks[order] = np.arange(len(a))
    # average tied ranks
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sorted_a[j + 1] == sorted_a[i]:
            j += 1
        if j > i:
            avg_rank = ranks[order[i:j + 1]].mean()
            ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def spearman(a, b):
    """Rank correlation — far less swayed by a single outlier magnitude
    than Pearson, which matters here: importance vectors are often
    dominated by one or two large components (see mean-KL rankings)."""
    return pearson(_rankdata(a), _rankdata(b))


def print_component_ranking(imp, labels, top_k=5, name=""):
    order = np.argsort(imp)[::-1]
    print(f"    top-{top_k} components by importance{f' ({name})' if name else ''}:")
    for idx in order[:top_k]:
        l, kind, h = labels[idx]
        tag = f"L{l}/head{h}" if kind == 'head' else f"L{l}/mlp"
        print(f"      {tag:<12} KL={imp[idx]:.5f}")


def total_variation_asymmetry(imp_D, imp_E):
    """
    sum|imp_D - imp_E| / sum(imp_D + imp_E) — a single permutation-
    invariant scalar in [0,1]: 0 = D and E draw on components with
    identical magnitude (fully shared), 1 = fully disjoint importance
    mass (fully dissociated). Unlike Pearson/Spearman on the whole
    vector, this doesn't require matching UP components by index across
    seeds (which is meaningless anyway — head identity isn't stable
    across independently-initialized models) and unlike the per-
    component `norm` below it isn't a ratio that blows up on near-zero
    components. This is the number to track across scale, not the
    per-component identity of who's "most D-specific."
    """
    imp_D, imp_E = np.asarray(imp_D), np.asarray(imp_E)
    den = np.sum(imp_D + imp_E)
    if den <= 0:
        return 0.0
    return float(np.sum(np.abs(imp_D - imp_E)) / den)


def print_differential_ranking(imp_D, imp_E, labels, top_k=5,
                               magnitude_floor_frac=0.02):
    """
    Surfaces which components are MOST D-specific vs MOST E-specific,
    with a magnitude floor: components where (imp_D + imp_E) is below
    magnitude_floor_frac of the largest such sum in the vector are
    excluded, because for near-zero components the normalized `norm`
    below is a noise ratio (0/eps type behavior), not a real signal —
    e.g. a component with imp_D=imp_E=0.00000 can print norm=+0.36
    purely from floating-point epsilon, which looks like "36% E-
    specific" but means nothing. `norm` is descriptive only — use
    total_variation_asymmetry() for anything you actually want to
    compare across scale/seed.
    """
    diff = imp_D - imp_E
    total = imp_D + imp_E
    floor = magnitude_floor_frac * total.max() if total.max() > 0 else 0.0
    keep = total >= floor
    norm = np.where(keep, diff / (total + 1e-8), 0.0)

    kept_idx = np.where(keep)[0]
    order_d = kept_idx[np.argsort(diff[kept_idx])[::-1]]
    order_e = kept_idx[np.argsort(diff[kept_idx])]

    def _row(idx):
        l, kind, h = labels[idx]
        tag = f"L{l}/head{h}" if kind == 'head' else f"L{l}/mlp"
        print(f"      {tag:<12} D={imp_D[idx]:.5f}  E={imp_E[idx]:.5f}  "
              f"Δ={diff[idx]:+.5f}  norm={norm[idx]:+.3f}")

    n_dropped = len(labels) - len(kept_idx)
    print(f"    most D-specific (imp_D - imp_E, highest first; "
          f"{n_dropped}/{len(labels)} near-zero components excluded):")
    for idx in order_d[:top_k]:
        _row(idx)
    print(f"    most E-specific (imp_E - imp_D, highest first):")
    for idx in order_e[:top_k]:
        _row(idx)


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6d — D-E CKIC, with generic-convergence control
# ══════════════════════════════════════════════════════════════════════

def ckic_de(model, target_subcomp='adj_agree_pre', control_subcomp='v_agree',
           n_examples=512, eval_seed=5678, mean_n=512, mean_seed=1234,
           verbose=True):
    """
    CKIC(D,E) on target_subcomp and control_subcomp for one model.
    v_agree is the ε≈0 control (both D and E succeed at it; sharing
    is expected regardless of scale) — a target CKIC drop that
    exceeds the control's drop is the ε-specific dissociation signal,
    same logic as the generic-convergence control used for the
    (failed) linear probe in Cell 11.5.
    """
    results = {}
    for sc in (target_subcomp, control_subcomp):
        imp_D, labels_D, n_D = ckic_importance_vector(
            model, 'D', sc, n_examples=n_examples, eval_seed=eval_seed,
            mean_n=mean_n, mean_seed=mean_seed)
        imp_E, labels_E, n_E = ckic_importance_vector(
            model, 'E', sc, n_examples=n_examples, eval_seed=eval_seed,
            mean_n=mean_n, mean_seed=mean_seed)
        if imp_D is None or imp_E is None:
            print(f"  [{sc}] missing subcomp positions for D or E — skipping")
            results[sc] = None
            continue
        assert labels_D == labels_E, "component ordering mismatch between D and E"

        r_pearson = pearson(imp_D, imp_E)
        r_spearman = spearman(imp_D, imp_E)
        tva = total_variation_asymmetry(imp_D, imp_E)
        results[sc] = {
            'imp_D': imp_D, 'imp_E': imp_E, 'labels': labels_D,
            'ckic': r_pearson, 'ckic_spearman': r_spearman,
            'tva': tva,
            'mean_imp_D': float(np.mean(imp_D)),
            'mean_imp_E': float(np.mean(imp_E)),
            'n_D': n_D, 'n_E': n_E,
        }
        if verbose:
            print(f"  [{sc}] n_positions: D={n_D}  E={n_E}   "
                  f"({len(imp_D)} components)")
            print(f"  [{sc}] CKIC(D,E) Pearson={r_pearson:+.3f}  "
                  f"Spearman={r_spearman:+.3f}  "
                  f"TV-asymmetry={tva:.3f}   "
                  f"(mean|imp_D|={np.mean(imp_D):.5f}, "
                  f"mean|imp_E|={np.mean(imp_E):.5f})")
            if abs(r_pearson - r_spearman) > 0.15:
                print(f"    NOTE: Pearson/Spearman disagree by "
                      f"{abs(r_pearson - r_spearman):.3f} — the Pearson "
                      f"number is likely being driven by one or two "
                      f"outlier-magnitude components rather than overall "
                      f"rank agreement. Trust Spearman more here.")
            if np.mean(imp_E) < 1e-5:
                print(f"    WARNING: E's mean importance on {sc} is ~0 — "
                      f"E may have NO allocated computation at this "
                      f"position (CKIC would be meaningless noise, not "
                      f"a low-sharing signal). Check mean|imp_E| on "
                      f"{control_subcomp} to confirm E has learnable "
                      f"circuitry the ablation methodology can detect "
                      f"at all before trusting this number.")
            print_component_ranking(imp_D, labels_D, name=f"D, {sc}")
            print_component_ranking(imp_E, labels_D, name=f"E, {sc}")
            print_differential_ranking(imp_D, imp_E, labels_D)
    return results


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6e — Scale sweep
# ══════════════════════════════════════════════════════════════════════

def ckic_sweep(sizes=("10K", "1M"), seeds=(0, 1),
              target_subcomp='adj_agree_pre', control_subcomp='v_agree',
              n_examples=512, eval_seed=5678, mean_n=512, mean_seed=1234):
    """
    Runs ckic_de at each (size, seed), then reports whether the
    target's CKIC drop across scale EXCEEDS the control's drop —
    mirrors the reporting structure of the (superseded)
    dissociation_with_control in Cell 11.5.
    """
    results = defaultdict(dict)

    for size in sizes:
        for seed in seeds:
            loaded = load_checkpoint(size, seed)
            if loaded is None:
                print(f"  [missing checkpoint: {size}/seed{seed} — skipping]")
                continue
            model, _log = loaded
            print(f"\n{size} / seed {seed}:")
            r = ckic_de(model, target_subcomp=target_subcomp,
                       control_subcomp=control_subcomp,
                       n_examples=n_examples, eval_seed=eval_seed,
                       mean_n=mean_n, mean_seed=mean_seed)
            results[(size, seed)] = r
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    print(f"\n=== CKIC scale trend: {target_subcomp} vs {control_subcomp} ===")
    for seed in seeds:
        r10 = results.get((sizes[0], seed))
        r1m = results.get((sizes[1], seed))
        if not r10 or not r1m:
            print(f"  seed {seed}: missing data, skipping")
            continue
        t10, t1m = r10.get(target_subcomp), r1m.get(target_subcomp)
        c10, c1m = r10.get(control_subcomp), r1m.get(control_subcomp)
        if not all([t10, t1m, c10, c1m]):
            print(f"  seed {seed}: missing target/control data, skipping")
            continue

        print(f"  seed {seed}:")
        for stat in ('ckic', 'ckic_spearman'):
            label = "Pearson " if stat == 'ckic' else "Spearman"
            t_delta = t1m[stat] - t10[stat]
            c_delta = c1m[stat] - c10[stat]
            specific = t_delta - c_delta
            print(f"    [{label}] {target_subcomp}: {t10[stat]:+.3f} -> "
                  f"{t1m[stat]:+.3f}  (Δ={t_delta:+.3f})   "
                  f"{control_subcomp}: {c10[stat]:+.3f} -> {c1m[stat]:+.3f}  "
                  f"(Δ={c_delta:+.3f})   ε-specific: {specific:+.3f}")

        # TV-asymmetry is a DISTANCE (0=shared, 1=disjoint) — dissociation
        # means it should INCREASE with scale, opposite sign convention
        # from the two correlation stats above. Permutation-invariant,
        # so this is the number I'd actually trust across seeds/scales.
        t_tva_delta = t1m['tva'] - t10['tva']
        c_tva_delta = c1m['tva'] - c10['tva']
        specific_tva = t_tva_delta - c_tva_delta
        print(f"    [TV-asym] {target_subcomp}: {t10['tva']:.3f} -> "
              f"{t1m['tva']:.3f}  (Δ={t_tva_delta:+.3f})   "
              f"{control_subcomp}: {c10['tva']:.3f} -> {c1m['tva']:.3f}  "
              f"(Δ={c_tva_delta:+.3f})   ε-specific: {specific_tva:+.3f}")

        # Verdict keyed off TV-asymmetry (permutation-invariant, not
        # outlier-ratio-driven) rather than either correlation stat.
        if specific_tva > 0.05:
            verdict = "[DISSOCIATION beyond generic drift — the N* signature]"
        elif abs(specific_tva) <= 0.05:
            verdict = "[flat relative to control — N* may sit outside swept range]"
        else:
            verdict = "[unexpected: target shares MORE than control with scale]"
        print(f"    verdict (TV-asymmetry-based): {verdict}")

    return results


# ══════════════════════════════════════════════════════════════════════
# CELL 11.6f — Diagnosis guide
# ══════════════════════════════════════════════════════════════════════

CKIC_DIAGNOSIS = """
=== CKIC Diagnosis Guide ===

BEFORE TRUSTING ANY CKIC NUMBER:

IF mean|imp_E| on adj_agree_pre is ~0 (see WARNING in ckic_de output):
  -> E has no ablatable computation at this position at all. CKIC is
     undefined, not "low sharing". Check mean|imp_E| on v_agree for
     the SAME model — if that's also ~0, something is wrong with the
     ablation methodology itself (or the model), not with sharing.
     If v_agree's mean|imp_E| is healthy but adj_agree_pre's isn't,
     that's itself informative (E has been allocated zero resources
     for this position) but needs to be reported as that finding, not
     folded into a CKIC number.

IF the per-component rankings (print_component_ranking) for D and E
share almost no top components even at low N:
  -> Consistent with genuine dissociation already at the smallest
     scale swept — N* may be below 10K for this pair. Consider adding
     a smaller size point.

IF CKIC(D,E) on adj_agree_pre is high AND CKIC on v_agree is ALSO
high, at every scale:
  -> Both flat. Check the specific-component delta (Δtarget - Δcontrol)
     before concluding no dissociation — a small but real ε-specific
     effect can be masked if control CKIC is also drifting for
     unrelated reasons (e.g. general capacity-driven reorganization
     unrelated to this pair).

HEADLINE PLOT target (same shape as the accuracy sweep, now for the
mechanistic half):
  -> adj_agree_pre CKIC(D,E): DECREASES with scale (or flat, with an
     explicit note that N* lies outside [10K, 1M]).
  -> v_agree CKIC(D,E): stays roughly flat/high across scale — the
     control confirms the drop above is subcomp-specific, not a
     generic representational drift with model capacity.

IF Pearson and Spearman CKIC disagree by more than ~0.15 (flagged
automatically in ckic_de's output):
  -> Pearson is being driven by one or two outlier-magnitude
     components (common here — one dominant MLP or head often
     dwarfs everything else). Trust the Spearman-based verdict in
     ckic_sweep's output over the raw Pearson number; investigate
     the specific outlier component via print_component_ranking
     before writing up either number.

AT 10K (only 6 components: n_layer=2 * (n_head=2 + 1)):
  -> Both target and control CKIC can look near-ceiling for a boring
     reason — MLPs may simply dominate over heads for ANY local
     token prediction in a 2-layer model, regardless of ε-specific
     sharing. Absolute 10K values are not strong evidence either way
     on their own; the DELTA relative to the control across scale is
     what the argument rests on. Do not report a bare "CKIC=0.85 at
     10K" as evidence of sharing without this caveat attached.

ALWAYS, per Lesson 5: check both seeds individually before reporting
a mean; sample-size caveats (Lesson 4) apply to n_examples here the
same way they did to accuracy — adj_agree_pre only fires on the
P_ADJ branch, so raise n_examples if the printed n_positions (D and
E, both now reported by ckic_de) looks thin, especially for L_E
whose adj_agree_pre positions require the same has_adj branch as
L_D's.
"""


# Example usage once checkpoints exist:
ckic_results = ckic_sweep(sizes=("10K", "1M"), seeds=(0, 1))

GPT: 79,008 params  (L=2 H=2 D=48)

10K / seed 0:
  [adj_agree_pre] n_positions: D=4939  E=4939   (6 components)
  [adj_agree_pre] CKIC(D,E) Pearson=+0.850  Spearman=+0.943  TV-asymmetry=0.589   (mean|imp_D|=2.02506, mean|imp_E|=0.52343)
    top-5 components by importance (D, adj_agree_pre):
      L0/mlp       KL=7.21406
      L1/mlp       KL=3.64519
      L1/head1     KL=1.26421
      L0/head1     KL=0.02343
      L0/head0     KL=0.00349
    top-5 components by importance (E, adj_agree_pre):
      L1/mlp       KL=1.73322
      L0/mlp       KL=1.40576
      L1/head1     KL=0.00107
      L0/head1     KL=0.00036
      L0/head0     KL=0.00013
    most D-specific (imp_D - imp_E, highest first; 3/6 near-zero components excluded):
      L0/mlp       D=7.21406  E=1.40576  Δ=+5.80830  norm=+0.674
      L1/mlp       D=3.64519  E=1.73322  Δ=+1.91197  norm=+0.355
      L1/head1     D=1.26421  E=0.00107  Δ=+1.26314  norm=+0.998
    most E-specific (imp_E - imp_D, highest first):
      L1/head1    